# STEP-2: Cell Type Annotation & Downstream Analysis

scRNA-seq analysis of CT26 mouse model with anti-VEGF/PD1 bispecific antibody treatment

Three groups: NC (control), AK112 (Akeso), RC148 (Remegen) - 2 replicates each

> **⚠️ DEG Analysis Warning:** With only 2 replicates per group, use log2FC for ranking genes, not p-values. Consider genes with |log2FC| > 1 as candidates for validation.

## Table of Contents

1. [Setup](#1-setup)
2. [Data Loading](#2-data-loading)
3. [Clustering](#3-clustering)
4. [Cell Type Annotation](#4-cell-type-annotation)
5. [Proportion Analysis](#5-proportion-analysis)
6. [VEGF/PD1 Target Analysis](#6-vegfpd1-target-analysis)
7. [Marker Gene Analysis](#7-marker-gene-analysis)
8. [CNV Analysis](#8-cnv-analysis)
9. [DEG Analysis - All 4 Comparisons](#9-deg-analysis)
10. [Pathway Enrichment](#10-pathway-enrichment)
11. [Export & Split](#11-export--split)

---

## 1. Setup

### 1.1 Imports

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import os
import scLucid as scl
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Arial'
import warnings
warnings.filterwarnings('ignore')

scl.setup_logging("INFO")
scl.set_figure_params(
    dpi=300,
    dpi_save=300,
    figsize=(6, 5),
    style="seaborn-v0_8",
    style_dict={'axes.grid': True},
    color_theme="default"
)

### 1.2 Paths

In [ ]:
# Define working directories
BASE_DIR = '/Users/luye/Library/Mobile Documents/com~apple~CloudDocs/Projects/Ongoing/202604JJH'
DATA_DIR = os.path.join(BASE_DIR, "1-DATA")
RESULTS_DIR = os.path.join(BASE_DIR, "2-OUTPUT")
os.makedirs(RESULTS_DIR, exist_ok=True)

### 1.3 Colors

In [ ]:
###** colors
color = {
    'PBS': '#BFCFE3',
    "R301":"#8B4C9D"
}

---

## 2. Data Loading

### 2.1 Load Preprocessed Data

In [ ]:
# --- Load Preprocessed Data ---
# Load preprocessed data
print("Loading preprocessed data...")
adata = sc.read_h5ad(os.path.join(DATA_DIR, "Step2-sce_preprocessed.h5ad"))
print(f"Dataset loaded: {adata.shape[0]} cells, {adata.shape[1]} genes")

print("\n---- Analysis Workflow ----")

In [ ]:
##====== Diagnostic UMAP before integration ======##
sc.pp.neighbors(adata, use_rep="X_pca", n_neighbors=15, n_pcs=40)
sc.tl.umap(adata, min_dist=0.2)

# 保存诊断用embedding，便于和整合后结果对比
adata.obsm["X_umap_pca"] = adata.obsm["X_umap"]

In [ ]:
sc.pl.embedding(adata, basis='X_umap_pca',
                color=['group', 'sampleID'],
                title=['Group', 'Sample'],
                size=10,
                ncols=2, wspace=0.5, hspace=0.3,
                show=False,frameon=False,
            )

---

## 3. Clustering

### 3.1 Visualize Embeddings

In [ ]:
adata.obs['group'] = pd.Categorical(
    adata.obs['group'], 
    categories=[
    'PBS','R301'
], 
    ordered=True
)

### 3.2 Sample-Group Crosstab

In [ ]:
ctab = pd.crosstab(adata.obs['sampleID'],adata.obs['group'],margins=True)
print(ctab)

### 3.3 Over-clustering

In [ ]:
# --- 3.2 Over-clustering ---
clustering_cfg = scl.al.ClusteringConfig(
    method="leiden",
    resolution=0.8,
    use_rep="X_pca",
    key_added="leiden_clusters" # Final cluster column name
)
adata = scl.al.cluster_cells(adata, config=clustering_cfg)

### 3.4 QC on UMAP

In [ ]:
sc.pl.embedding(adata, basis='X_umap_pca',
                color=['pct_counts_mt', 'doubletdetection_score','heuristic_confidence_score'],
                ncols=3, wspace=0.5, hspace=0.3,
                show=False,frameon=False,cmap='viridis'
            )

In [ ]:
sc.pl.embedding(adata, basis='X_umap_pca',
                color=['leiden_clusters'],
                size=9,
                legend_loc="on data",
                legend_fontsize=14,
                #ncols=3, wspace=0.5, hspace=0.3,
                show=False,frameon=False,
            )

### 3.5 Save

In [ ]:
adata.write(DATA_DIR+ '/Step3-sce_annotated.h5ad', compression='gzip')

---

## 4. Cell Type Annotation

### 4.1 Load Data

In [ ]:
adata = sc.read_h5ad(os.path.join(DATA_DIR, "Step3-sce_annotated.h5ad"))
adata

In [ ]:
import gc 
gc.collect()

### 4.2 Coarse Marker Filter

In [ ]:
# ==============================================================================
# Step 2: Marker discovery (Marker Discovery & Enrichment)
# ==============================================================================
print("\n##====== 2. Marker Discovery & Enrichment ======##")
# --- 4.2 Coarse marker filter ---
# No strict filtering yet，目的是为富集分析提供尽可能多的基因
de_config = scl.al.DifferentialConfig(
    groupby='leiden_clusters',
    use_raw=True,
    pval_cutoff=0.05
)
scl.al.find_markers(adata, config=de_config)

### 4.3 Fine Marker Filter

In [ ]:
# --- 4.3 Filter high-specificity markers ---
# 现在，我们应用严格的过滤来找到用于注释和可视化的“金标准”marker
filter_cfg = scl.al.FilterMarkersConfig(
    key="rank_genes_groups",              
    key_added="highly_specific_markers_df", # Final filtered result
    min_log2fc=0.5,
    max_padj=0.01,
    min_in_group_pct=0.2,
    max_out_group_pct=None,
    min_diff_pct=0.1, # Min 10% difference
    keep_top_n=50
)
highly_specific_markers_df = scl.al.filter_markers(adata, config=filter_cfg)

### 4.4 Visualize Markers

In [ ]:
# --- 4.4 Visualize markers ---
scl.al.visualize_markers(
    adata,
    markers=highly_specific_markers_df, # Pass filtered DataFrame
    groupby="leiden_clusters",
    plot_type="dotplot",
    swap_axes=True,
    n_genes_per_group=3 # Top 5 per cluster
)

### 4.5 Rank Gene Groups

In [ ]:
sc.pl.rank_genes_groups(adata, n_genes=25, sharey=False)

### 4.6 Enrichment

In [ ]:
# --- 4.5 Enrichment analysis ---
# Enrichment sensitivity note，使用过滤前的marker列表效果可能更好
enrichment_config = scl.al.EnrichmentConfig(
    de_key='rank_genes_groups_df', # Use default output
    mode='offline',
    organism='mouse',
    gene_sets_offline=['go_bp', 'reactome'] # 可以是一个列表
)
scl.al.run_enrichment(adata, groupby='leiden_clusters', config=enrichment_config)

### 4.7 Export Summary

In [ ]:
# --- 2.5 导出整合了“精筛Marker”和“富集结果”的摘要 ---
summary_path = os.path.join(RESULTS_DIR, "al_annotation/annotation_summary.md")
scl.al.summarize_markers_and_enrichment(
    adata,
    groupby="leiden_clusters",
    markers_df=highly_specific_markers_df, # Use filtered markers
    enrichment_key="enrichment",            # Auto-load from .uns
    n_markers=25,
    n_terms=10,
    summary_file=summary_path
)

### CNV Analysis

In [ ]:
# ==============================================================================
# CNV Analysis - use raw counts (CopyKAT-like pipeline)
# ==============================================================================

import os
import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import issparse
from scipy.ndimage import uniform_filter1d

# ==============================================================================
# 预备工作：创建输出目录
# 请确保 RESULTS_DIR, DATA_DIR 和 adata 在外部已定义
# ==============================================================================
CNV_DIR = os.path.join(RESULTS_DIR, "cnv_analysis")
os.makedirs(CNV_DIR, exist_ok=True)

print("="*80)
print("Starting CopyKAT-like CNV analysis")
print(f"Results will be saved to: {CNV_DIR}")
print("="*80)

# ==============================================================================
# Step 0: 重新读取原始count数据
# ==============================================================================
print("\nStep 0: Loading original count data...")

# 请根据你的实际情况修改路径
raw_data_path = os.path.join(DATA_DIR, 'Step1-sce_cleaned.h5ad')  # 修改为你的实际路径
print(f"Reading from: {raw_data_path}")

if not os.path.exists(raw_data_path):
    print(f"ERROR: File not found: {raw_data_path}")
    print("\nPlease provide the path to your original count data.")
    raise FileNotFoundError(f"Raw count data not found at {raw_data_path}")

adata_counts = sc.read_h5ad(raw_data_path)
print(f"Loaded raw counts: {adata_counts.shape[0]} cells × {adata_counts.shape[1]} genes")

# 验证数据是否为原始counts
print("\nVerifying data type...")
if issparse(adata_counts.X):
    data_sample = adata_counts.X[:100, :100].toarray()
else:
    data_sample = adata_counts.X[:100, :100]

max_val = data_sample.max()
has_decimals = np.any(data_sample % 1 != 0)

if max_val < 20 or has_decimals:
    print("\n⚠️  WARNING: Data appears to be normalized/log-transformed!")
    print("Attempting to find counts in layers...")
    if 'counts' in adata_counts.layers:
        print("Found 'counts' layer, using it...")
        adata_counts.X = adata_counts.layers['counts'].copy()
    elif 'raw_counts' in adata_counts.layers:
        print("Found 'raw_counts' layer, using it...")
        adata_counts.X = adata_counts.layers['raw_counts'].copy()
    else:
        raise ValueError("Cannot proceed without raw count data. Please provide the original count matrix.")
else:
    print("✓ Data appears to be raw counts")

# ==============================================================================
# Step 1: 匹配细胞并筛选
# ==============================================================================
print("\nStep 1: Matching cells with processed data...")
current_cells = adata.obs.index  # adata是你当前处理到的数据

common_cells = adata_counts.obs.index.intersection(current_cells)
print(f"Common cells: {len(common_cells)} / {len(current_cells)}")

if len(common_cells) == 0:
    raise ValueError("Cell barcodes don't match. Please check data sources.")

adata_counts = adata_counts[common_cells, :].copy()

print("\nTransferring metadata from processed data...")
metadata_cols = ['leiden_clusters', 'sampleID', 'cell_type']  # 根据需要修改
for col in metadata_cols:
    if col in adata.obs.columns:
        adata_counts.obs[col] = adata.obs.loc[common_cells, col]
        print(f"  Transferred: {col}")

# ==============================================================================
# Step 2: 添加基因组坐标
# ==============================================================================
print("\nStep 2: Adding genomic coordinates...")
gene_info_path = '/Users/luye/Data/Sig/gene_info_mmu.txt'  # 请确保此路径正确
gene_info = pd.read_csv(gene_info_path, sep='\t')

gene_info = gene_info.rename(columns={
    'chromosome_name': 'chromosome',
    'start_position': 'start',
    'end_position': 'end',
    'external_gene_name': 'gene_name'
})

gene_info = gene_info[gene_info['gene_biotype'] == 'protein_coding'].copy()
gene_info = gene_info.drop_duplicates(subset=['gene_name'], keep='first')
gene_info = gene_info.set_index('gene_name')

adata_counts.var['chromosome'] = gene_info.reindex(adata_counts.var_names)['chromosome']
adata_counts.var['start'] = gene_info.reindex(adata_counts.var_names)['start']
adata_counts.var['end'] = gene_info.reindex(adata_counts.var_names)['end']

initial_genes = adata_counts.n_vars
adata_counts = adata_counts[:, adata_counts.var[['chromosome', 'start', 'end']].notna().all(axis=1)].copy()
print(f"Genes with genomic coordinates: {adata_counts.n_vars} / {initial_genes}")

adata_counts.var['chromosome'] = adata_counts.var['chromosome'].astype(str)
standard_chroms = [str(i) for i in range(1, 20)] + ['X', 'Y']
adata_counts = adata_counts[:, adata_counts.var['chromosome'].isin(standard_chroms)].copy()
print(f"Standard chromosomes: {adata_counts.n_vars} genes")

# ==============================================================================
# Step 3: 基因过滤 - 基于原始counts
# ==============================================================================
print("\nStep 3: Filtering genes based on expression...")
if issparse(adata_counts.X):
    gene_counts = np.array(adata_counts.X.sum(axis=0)).flatten()
    cells_per_gene = np.array((adata_counts.X > 0).sum(axis=0)).flatten()
else:
    gene_counts = adata_counts.X.sum(axis=0)
    cells_per_gene = (adata_counts.X > 0).sum(axis=0)

min_cells = int(0.05 * adata_counts.n_obs)
gene_filter = (cells_per_gene >= min_cells) & (gene_counts >= min_cells)
adata_counts = adata_counts[:, gene_filter].copy()

chr_counts = adata_counts.var['chromosome'].value_counts()
valid_chroms = chr_counts[chr_counts >= 100].index.tolist()
adata_counts = adata_counts[:, adata_counts.var['chromosome'].isin(valid_chroms)].copy()

adata_counts.var['chromosome'] = pd.Categorical(
    adata_counts.var['chromosome'],
    categories=[str(i) for i in range(1, 20)] + ['X', 'Y'],
    ordered=True
)
adata_counts = adata_counts[:, adata_counts.var.sort_values(['chromosome', 'start']).index].copy()
print(f"Final gene count after filtering: {adata_counts.n_vars}")

# ==============================================================================
# Step 4: 定义参考细胞 (Helper robust MAD)
# ==============================================================================
def mad(x, axis=0, eps=1e-6):
    med = np.median(x, axis=axis)
    mad_val = np.median(np.abs(x - np.expand_dims(med, axis=axis)), axis=axis)
    return mad_val + eps

# ==============================================================================
# Step 5: 选择参考细胞群
# ==============================================================================
print("\n[CNV] Step 5: Defining reference cells...")
REF_CLUSTERS = ['6','7','8','9','13','14','15','17','20','21','22','23']  # T and NK clusters in your data

if 'leiden_clusters' not in adata_counts.obs.columns:
    raise ValueError("leiden_clusters not found in adata_counts.obs")

leiden_str = adata_counts.obs['leiden_clusters'].astype(str)
ref_mask = leiden_str.isin(REF_CLUSTERS).values
n_ref = ref_mask.sum()
if n_ref < 200:
    print(f"⚠️  Reference cells too few ({n_ref}). Consider adding more immune clusters.")

adata_counts.obs['cnv_is_reference'] = ref_mask
print(f"[CNV] Reference cells: {n_ref} / {adata_counts.n_obs} ({n_ref/adata_counts.n_obs*100:.1f}%)")

# ==============================================================================
# Step 6: 过滤线粒体/核糖体/细胞周期等易产生伪影的基因
# ==============================================================================
print("\n[CNV] Step 6: Filtering genes prone to artifacts (mt/ribo/cell-cycle)...")
symbol_cols = [c for c in ['gene_name', 'gene_symbol', 'symbol', 'external_gene_name'] if c in adata_counts.var.columns]

if len(symbol_cols) > 0:
    sym_col = symbol_cols[0]
    gene_symbols = adata_counts.var[sym_col].astype(str)
else:
    # 强制将 var_names 转换为 Pandas Series，以保持后续方法返回类型的一致性
    gene_symbols = pd.Series(adata_counts.var_names, index=adata_counts.var_names).astype(str)

mt_mask = gene_symbols.str.match(r'^(mt-|MT-|Mt-)', na=False)
ribo_mask = gene_symbols.str.match(r'^(Rpl|Rps|RPL|RPS)', na=False)

cell_cycle_genes = set()  
cc_mask = gene_symbols.isin(cell_cycle_genes) if len(cell_cycle_genes) > 0 else np.zeros(adata_counts.n_vars, dtype=bool)

# 使用 np.asarray() 替代 .values，实现安全的类型转换和按位运算
drop_mask = (np.asarray(mt_mask) | np.asarray(ribo_mask) | np.asarray(cc_mask))

adata_counts.var['cnv_mt'] = np.asarray(mt_mask)
adata_counts.var['cnv_ribo'] = np.asarray(ribo_mask)
adata_counts.var['cnv_cc'] = np.asarray(cc_mask)

print(f"[CNV] Dropping mt: {np.sum(mt_mask)}, ribo: {np.sum(ribo_mask)}, cell-cycle: {np.sum(cc_mask)}")
adata_cnv = adata_counts[:, ~drop_mask].copy()

# ==============================================================================
# Step 7: 标准化 Counts (log1p CPM-like)
# ==============================================================================
print("\n[CNV] Step 7: Normalizing counts...")
sc.pp.normalize_total(adata_cnv, target_sum=1e4)
sc.pp.log1p(adata_cnv)

X = adata_cnv.X
if issparse(X):
    X = X.toarray()
else:
    X = X.copy()

# ==============================================================================
# Step 8: 染色体级别的平滑处理 (Vectorized)
# ==============================================================================
print("\n[CNV] Step 8: Chromosome-wise smoothing (vectorized)...")
chromosomes = adata_cnv.var['chromosome'].astype(str).values
unique_chroms = pd.unique(chromosomes)

window_size = 101
X_smooth = np.zeros_like(X, dtype=np.float32)

for chrom in unique_chroms:
    idx = np.where(chromosomes == chrom)[0]
    if len(idx) < 30:
        X_smooth[:, idx] = X[:, idx]
        continue
    X_chr = X[:, idx]
    X_chr_smooth = uniform_filter1d(X_chr, size=window_size, axis=1, mode='nearest')
    X_smooth[:, idx] = X_chr_smooth

# ==============================================================================
# Step 9: 匹配参考细胞进行稳健缩放 (MAD)
# ==============================================================================
print("\n[CNV] Step 9: Centering to reference and robust scaling (MAD)...")
ref_mask_cnv = adata_cnv.obs['cnv_is_reference'].values
if ref_mask_cnv.sum() < 50:
    raise ValueError("Too few reference cells after filtering.")

X_ref = X_smooth[ref_mask_cnv, :]
ref_mean = X_ref.mean(axis=0)
ref_mad = mad(X_ref, axis=0)

Z = (X_smooth - ref_mean) / ref_mad
Z = np.clip(Z, -10, 10)

# ==============================================================================
# Step 10: 计算 CNV Scores
# ==============================================================================
print("\n[CNV] Step 10: Computing robust CNV scores...")
cnv_chr_scores = {}
for chrom in unique_chroms:
    idx = np.where(chromosomes == chrom)[0]
    if len(idx) < 30:
        continue
    cnv_chr_scores[f'chr{chrom}_score'] = np.median(np.abs(Z[:, idx]), axis=1)

cnv_chr_df = pd.DataFrame(cnv_chr_scores, index=adata_cnv.obs_names)

adata_cnv.obs['cnv_score'] = cnv_chr_df.median(axis=1).values
adata_cnv.obs['cnv_extreme_frac'] = (np.abs(Z) > 2.0).mean(axis=1)

for c in cnv_chr_df.columns:
    adata_cnv.obs[c] = cnv_chr_df[c].values

# ==============================================================================
# Step 11: 非整倍体预测 (Conservative threshold + optional GMM)
# ==============================================================================
print("\n[CNV] Step 11: Predicting aneuploid cells (conservative)...")
ref_scores = adata_cnv.obs.loc[ref_mask_cnv, 'cnv_score'].values
ref_med = np.median(ref_scores)
ref_mad_score = np.median(np.abs(ref_scores - ref_med)) + 1e-6

thr = ref_med + 3.0 * ref_mad_score
adata_cnv.obs['predicted_class_thr'] = np.where(adata_cnv.obs['cnv_score'] > thr, 'aneuploid', 'diploid')

print(f"[CNV] Reference cnv_score median={ref_med:.4f}, MAD={ref_mad_score:.4f}, threshold={thr:.4f}")

try:
    from sklearn.mixture import GaussianMixture
    feats = adata_cnv.obs[['cnv_score', 'cnv_extreme_frac']].values
    gmm = GaussianMixture(n_components=2, random_state=42)
    gmm_labels = gmm.fit_predict(feats)
    means = [adata_cnv.obs['cnv_score'].values[gmm_labels == i].mean() for i in [0,1]]
    aneu_comp = int(np.argmax(means))
    adata_cnv.obs['predicted_class_gmm'] = np.where(gmm_labels == aneu_comp, 'aneuploid', 'diploid')
except Exception as e:
    print("[CNV] GMM skipped due to:", repr(e))

adata_cnv.obs['predicted_class'] = adata_cnv.obs['predicted_class_thr']

# ==============================================================================
# Step 12: 简单的分布统计并保存 CSV
# ==============================================================================
print("\n[CNV] Summary by sample and cluster...")
if 'sampleID' in adata_cnv.obs.columns:
    sample_tab = (adata_cnv.obs
                  .groupby('sampleID')['predicted_class']
                  .value_counts(normalize=True)
                  .rename('frac')
                  .reset_index())
    print(sample_tab.pivot(index='sampleID', columns='predicted_class', values='frac').fillna(0).round(3))

adata_cnv.obs.to_csv(os.path.join(CNV_DIR, "cnv_predictions_improved.csv"))

# ==============================================================================
# Step 13: Visualizations (revised & improved with percentages)
# ==============================================================================
print("\nStep 13: Generating visualizations (revised)...")

# 1) CNV score distribution
print("\n1) CNV score distribution...")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(adata_cnv.obs.loc[ref_mask_cnv, 'cnv_score'], bins=50,
             alpha=0.6, label='Reference', color='#3498db', density=True)
axes[0].hist(adata_cnv.obs.loc[~ref_mask_cnv, 'cnv_score'], bins=50,
             alpha=0.6, label='Non-reference', color='#e74c3c', density=True)
axes[0].axvline(thr, color='black', linestyle='--', linewidth=2, label=f'Thr={thr:.3f}')
axes[0].set_xlabel('CNV Score')
axes[0].set_ylabel('Density')
axes[0].set_title('CNV Score: Reference vs Others')
axes[0].legend()
axes[0].grid(alpha=0.3)

for pred_class, color in [('diploid', '#2ecc71'), ('aneuploid', '#e74c3c')]:
    mask = (adata_cnv.obs['predicted_class'] == pred_class).values
    if mask.sum() == 0: continue
    axes[1].hist(adata_cnv.obs.loc[mask, 'cnv_score'], bins=50,
                 alpha=0.6, label=f'{pred_class} (n={mask.sum()})', density=True, color=color)
axes[1].axvline(thr, color='black', linestyle='--', linewidth=2, label=f'Thr={thr:.3f}')
axes[1].set_xlabel('CNV Score')
axes[1].set_ylabel('Density')
axes[1].set_title('CNV Score by Predicted Class')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
fig_file = os.path.join(CNV_DIR, "cnv_score_distribution.pdf")
plt.savefig(fig_file, dpi=300, bbox_inches='tight')
plt.show()

# 2) UMAP overlays
print("\n2) UMAP overlays...")
if 'X_umap' not in adata_cnv.obsm.keys():
    if 'X_umap' in adata.obsm.keys() and np.all(adata_cnv.obs_names == adata.obs_names):
        adata_cnv.obsm['X_umap'] = adata.obsm['X_umap'].copy()
    else:
        print("⚠️  Computing UMAP on adata_cnv (may not match your main embedding).")
        sc.pp.pca(adata_cnv, n_comps=50)
        sc.pp.neighbors(adata_cnv)
        sc.tl.umap(adata_cnv)

sc.pl.umap(
    adata_cnv,
    color=['cnv_score', 'cnv_extreme_frac', 'predicted_class'],
    wspace=0.4,
    show=True
)

# 3) Per-chromosome CNV scores (box/violin)
print("\n3) CNV scores per chromosome...")
chr_score_cols = [c for c in adata_cnv.obs.columns if c.startswith('chr') and c.endswith('_score')]
if len(chr_score_cols) > 0:
    df = adata_cnv.obs[chr_score_cols + ['predicted_class']].copy()
    long = df.melt(id_vars='predicted_class', var_name='chrom', value_name='cnv_chr_score')
    long['chrom'] = (long['chrom'].str.replace('chr', '', regex=False).str.replace('_score', '', regex=False))
    
    chrom_order = [str(i) for i in range(1, 20)] + ['X', 'Y']
    long['chrom'] = pd.Categorical(long['chrom'], categories=chrom_order, ordered=True)
    long = long.sort_values('chrom')

    fig, ax = plt.subplots(figsize=(16, 5))
    sns.violinplot(
        data=long, x='chrom', y='cnv_chr_score', hue='predicted_class',
        split=True, inner='quartile', palette={'diploid': '#2ecc71', 'aneuploid': '#e74c3c'}, ax=ax, cut=0
    )
    ax.set_xlabel('Chromosome')
    ax.set_ylabel('Median(|Z|) per chromosome')
    ax.set_title('Per-chromosome CNV score by predicted class')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(CNV_DIR, "cnv_scores_per_chromosome_violin.pdf"), dpi=300, bbox_inches='tight')
    plt.show()

# 4) Aneuploid proportion by sample & cluster
print("\n4) Aneuploid proportion by sample and cluster...")
def aneu_stats(group_col):
    if group_col not in adata_cnv.obs.columns: return None
    tab = adata_cnv.obs.groupby(group_col)['predicted_class'].value_counts(normalize=False).unstack(fill_value=0)
    tab['total'] = tab.sum(axis=1)
    if 'aneuploid' not in tab.columns: tab['aneuploid'] = 0
    tab['aneuploid_pct'] = tab['aneuploid'] / tab['total'] * 100
    return tab.sort_values('aneuploid_pct', ascending=True)

sample_stats = aneu_stats('sampleID')
cluster_stats = aneu_stats('leiden_clusters')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

if sample_stats is not None:
    top = sample_stats
    axes[0].barh(top.index.astype(str), top['aneuploid_pct'], color='#e74c3c', height=0.6)
    axes[0].set_xlabel('Aneuploid %')
    axes[0].set_title('Aneuploid proportion by sample')
    max_val = top['aneuploid_pct'].max()
    axes[0].set_xlim(0, max_val * 1.15 if max_val > 0 else 100)
    axes[0].grid(axis='x', alpha=0.3)
    for i, v in enumerate(top['aneuploid_pct']):
        axes[0].text(v + (max_val * 0.01), i, f"{v:.1f}%", va='center', color='black', fontsize=10, fontweight='bold')
else:
    axes[0].set_axis_off(); axes[0].set_title("No sampleID in adata_cnv.obs")

if cluster_stats is not None:
    top = cluster_stats.tail(25)
    axes[1].barh(top.index.astype(str), top['aneuploid_pct'], color='#e74c3c', height=0.6)
    axes[1].set_xlabel('Aneuploid %')
    axes[1].set_title('Aneuploid proportion by cluster (top 25)')
    max_val = top['aneuploid_pct'].max()
    axes[1].set_xlim(0, max_val * 1.15 if max_val > 0 else 100)
    axes[1].grid(axis='x', alpha=0.3)
    for i, v in enumerate(top['aneuploid_pct']):
        axes[1].text(v + (max_val * 0.01), i, f"{v:.1f}%", va='center', color='black', fontsize=10, fontweight='bold')
else:
    axes[1].set_axis_off(); axes[1].set_title("No leiden_clusters in adata_cnv.obs")

plt.tight_layout()
plt.savefig(os.path.join(CNV_DIR, "aneuploid_proportion_by_group.pdf"), dpi=300, bbox_inches='tight')
plt.show()

# 5) CNV heatmap along genome
print("\n5) CNV heatmap along genome (sampled cells)...")
np.random.seed(42)
n_show = min(800, adata_cnv.n_obs)
show_cells = np.random.choice(adata_cnv.obs_names, size=n_show, replace=False)

show_obs = adata_cnv.obs.loc[show_cells].copy()
show_obs['is_ref'] = show_obs.index.map(lambda x: adata_cnv.obs.loc[x, 'cnv_is_reference'])
show_obs = show_obs.sort_values(['is_ref', 'cnv_score'], ascending=[False, True])
show_cells = show_obs.index.values

cell_idx = adata_cnv.obs_names.get_indexer(show_cells)
Z_show = Z[cell_idx, :]

fig, ax = plt.subplots(figsize=(18, 7))
im = ax.imshow(Z_show, aspect='auto', cmap='bwr', vmin=-3, vmax=3, interpolation='nearest')
ax.set_xlabel('Genes ordered by chromosome & position')
ax.set_ylabel('Sampled cells (ref first, then low→high cnv_score)')
ax.set_title('CNV Z heatmap (centered to reference, MAD-scaled)')
cbar = plt.colorbar(im, ax=ax, fraction=0.02, pad=0.01)
cbar.set_label('Z')

chrom = np.array(chromosomes)
boundaries = np.where(chrom[:-1] != chrom[1:])[0] + 0.5
for b in boundaries:
    ax.axvline(b, color='k', linewidth=0.3, alpha=0.4)

plt.tight_layout()
plt.savefig(os.path.join(CNV_DIR, "cnv_heatmap_sampled.pdf"), dpi=300, bbox_inches='tight')
plt.show()

print("\n" + "="*80)
print("CNV visualization completed!")
print("="*80)

In [ ]:
# ==============================================================================
# Transfer results to adata
# ==============================================================================
print("\nTransferring CNV results back to original adata...")

# Transfer CNV results to original adata
cnv_cols = ['cnv_score', 'predicted_class']
#cnv_cols += [col for col in adata_norm.obs.columns if col.startswith('chr') and col.endswith('_score')]

for col in cnv_cols:
    adata.obs[col] = adata_cnv.obs.loc[adata.obs.index, col]

print(f"Transferred {len(cnv_cols)} CNV columns to original adata")

print("\nCNV Analysis Summary:")
print("="*80)
print(f"Total cells analyzed: {adata.n_obs}")
print(f"Predicted aneuploid cells: {(adata.obs['predicted_class'] == 'aneuploid').sum()}")
print(f"Predicted diploid cells: {(adata.obs['predicted_class'] == 'diploid').sum()}")
print(f"Mean CNV score: {adata.obs['cnv_score'].mean():.4f}")
print("="*80)

print("\nCNV analysis completed!")
print(f"All results saved to: {CNV_DIR}")

In [ ]:
adata.write(DATA_DIR+ '/Step3-sce_annotated.h5ad', compression='gzip')

### 4.8 Manual Refinement

In [ ]:
# Define mapping file path
mapping_file_path = os.path.join(RESULTS_DIR, "al_annotation/manual_mapping.xlsx")

# Call mapping function
scl.al.apply_annotation_mapping(
    adata,
    cluster_key="leiden_clusters",       # Specify cluster column
    mapping=mapping_file_path,           # Pass file path
    key_added="celltype"         # Define new column name
)
print("Annotation complete:")
print(adata.obs['celltype'].value_counts())

In [ ]:
#adata = adata[~adata.obs["leiden_clusters"].isin(["4","10","22"]), :]
adata

### 4.9 Set Categorical Order

In [ ]:
adata.obs['group'] = adata.obs['group'].cat.reorder_categories(['PBS','R301'], ordered=True)

In [ ]:
celltype_colors = {
    # Malignant epithelial/tumor states
    "Mesenchymal-like malignant cells": "#B2182B",
    "Growth factor-high malignant cells": "#D6604D",
    "Translation-high malignant cells": "#F4A582",
    "Hypoxic/glycolytic malignant cells": "#CA0020",
    "Cycling malignant cells": "#8B0000",
    "IFN-stimulated malignant cells": "#E78AC3",
    "Proliferating malignant cells": "#67001F",
    "CDH13+ mesenchymal-like malignant cells": "#A50F15",

    # Myeloid cells
    "Neutrophils": "#FDB863",
    "IL1B+ inflammatory macrophages": "#2166AC",
    "CCL2+ inflammatory macrophages": "#4393C3",
    "C1q+ antigen-presenting macrophages": "#92C5DE",
    "MRC1+ resident-like macrophages": "#053061",
    "CTSK+ osteoclast-like macrophages": "#5E3C99",

    # Lymphoid cells
    "Cytotoxic NK/NKT-like cells": "#1B9E77",
    "Cytotoxic CD8 T cells": "#66A61E",
    "Activated T cells": "#B2DF8A",
    "B cells": "#7570B3",

    # Dendritic cells
    "pDCs": "#E6AB02",
    "CLEC9A+ cDCs": "#A6761D",
    "CCR7+ mature DCs": "#FFD92F",

    # Stromal / vascular / mast
    "ECM-remodeling fibroblasts": "#A65628",
    "Vascular endothelial cells": "#4DAF4A",
    "Mast cells": "#F781BF"
}
celltype_order = [
    # Malignant compartment
    "Mesenchymal-like malignant cells",
    "Growth factor-high malignant cells",
    "Translation-high malignant cells",
    "Hypoxic/glycolytic malignant cells",
    "Cycling malignant cells",
    "IFN-stimulated malignant cells",
    "Proliferating malignant cells",
    "CDH13+ mesenchymal-like malignant cells",

    # Myeloid compartment
    "Neutrophils",
    "IL1B+ inflammatory macrophages",
    "CCL2+ inflammatory macrophages",
    "C1q+ antigen-presenting macrophages",
    "MRC1+ resident-like macrophages",
    "CTSK+ osteoclast-like macrophages",

    # T / NK / B cells
    "Cytotoxic NK/NKT-like cells",
    "Cytotoxic CD8 T cells",
    "Activated T cells",
    "B cells",

    # DCs
    "pDCs",
    "CLEC9A+ cDCs",
    "CCR7+ mature DCs",

    # Stromal / vascular / mast
    "ECM-remodeling fibroblasts",
    "Vascular endothelial cells",
    "Mast cells"
]

In [ ]:
# Set categorical order
adata.obs['celltype'] = adata.obs['celltype'].astype('category')
adata.obs['celltype'] = adata.obs['celltype'].cat.reorder_categories(celltype_order, ordered=True)

### 4.10 Marker Gene Analysis

In [ ]:
# Calculate cluster markers
# Use t-test or Wilcoxon test
sc.tl.rank_genes_groups(adata, groupby='celltype', method="wilcoxon", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(
    adata, groupby="celltype", standard_scale="var", n_genes=5, key="rank_genes_groups"
)

In [ ]:
sc.pl.rank_genes_groups(adata, n_genes=25, sharey=False)

In [ ]:
marker_dict = {
    # -------------------------
    # Malignant cell states
    # -------------------------
    "Mesenchymal-like malignant cells": [
        "Cdh13", "Slit2", "Sema3a", "Hmga2", "Prrx1", "Vcam1", "Col1a2"
    ],

    "Growth factor-high malignant cells": [
        "Ereg", "Hbegf", "Grem1", "Timp1", "Nes", "Itga6", "Cald1"
    ],

    "Translation-high malignant cells": [
        "Eif4a1", "Eif3a", "Eif3e", "Eef1a1", "Eef2", "Rplp0", "Rps3", "Rps6"
    ],

    "Hypoxic/glycolytic malignant cells": [
        "P4ha2", "Plod2", "Ankrd37", "Bnip3", "Ndrg1", "Pdk1", "Higd1a", "Eno2"
    ],

    "Cycling malignant cells": [
        "Mki67", "Top2a", "Prc1", "Aurkb", "Cenpf", "Kif11", "Ube2c"
    ],

    "IFN-stimulated malignant cells": [
        "Ifit1", "Ifit3", "Usp18", "Isg15", "Oasl2", "Gbp2", "Rsad2", "Ifih1"
    ],

    "Proliferating malignant cells": [
        "Cenpf", "Ube2c", "Cdc20", "Plk1", "Ccnb1", "Aurka", "Tpx2", "Kif20a"
    ],

    "CDH13+ mesenchymal-like malignant cells": [
        "Cdh13", "Slit2", "Sema3a", "Zfpm2", "Nav2", "Ghr", "Hmga2"
    ],


    # -------------------------
    # Myeloid cells
    # -------------------------
    "Neutrophils": [
        "S100a8", "S100a9", "Csf3r", "Trem1", "Cxcl2", "Il1b", "G0s2", "Hdc"
    ],

    "IL1B+ inflammatory macrophages": [
        "Il1b", "Ccr2", "Cd14", "Itgam", "Cd86", "Clec7a", "Cybb", "Lyz2"
    ],

    "CCL2+ inflammatory macrophages": [
        "Ccl2", "Ccl6", "Ccl7", "Ccl9", "Cxcl2", "Mafb", "Cd68", "Mpeg1"
    ],

    "C1q+ antigen-presenting macrophages": [
        "C1qa", "C1qb", "C1qc", "Mrc1", "Apoe", "Ctss", "Lgmn", "Trem2"
    ],

    "MRC1+ resident-like macrophages": [
        "Mrc1", "Gas6", "F13a1", "Apoe", "Selenop", "Stab1", "Csf1r", "Colec12"
    ],

    "CTSK+ osteoclast-like macrophages": [
        "Ctsk", "Acp5", "Atp6v0d2", "Mmp9", "Tnfrsf11a", "Nfatc1", "Mmp14", "Lpl"
    ],


    # -------------------------
    # Lymphoid cells
    # -------------------------
    "Cytotoxic NK/NKT-like cells": [
        "Ncr1", "Nkg7", "Gzmb", "Prf1", "Klrd1", "Ctsw", "Il2rb", "Txk"
    ],

    "Cytotoxic CD8 T cells": [
        "Cd8a", "Cd8b1", "Cd3d", "Cd3e", "Trac", "Pdcd1", "Lag3", "Nkg7", "Prf1"
    ],

    "Activated T cells": [
        "Icos", "Cd2", "Cd3d", "Cd3e", "Cd28", "Cd247", "Itk", "Lck"
    ],

    "B cells": [
        "Cd79a", "Cd79b", "Ms4a1", "Pax5", "Ebf1", "Ighd", "Ighm", "Igkc"
    ],


    # -------------------------
    # Dendritic cells
    # -------------------------
    "pDCs": [
        "Siglech", "Ccr9", "Tcf4", "Irf8", "Lair1", "Rnase6", "Tmem229b"
    ],

    "CLEC9A+ cDCs": [
        "Clec9a", "Wdfy4", "Flt3", "Irf8", "Cd24a", "H2-Aa", "H2-Eb1", "Cd74"
    ],

    "CCR7+ mature DCs": [
        "Ccr7", "Ccl22", "Flt3", "Cd83", "Mreg", "Lsp1", "Itga4", "Plxnc1"
    ],


    # -------------------------
    # Stromal / vascular / mast cells
    # -------------------------
    "ECM-remodeling fibroblasts": [
        "Col1a1", "Col3a1", "Col5a2", "Col6a2", "Col6a3", "Fbn1", "Mmp2", "Loxl2"
    ],

    "Vascular endothelial cells": [
        "Emcn", "Ptprb", "Esam", "Egfl7", "Cdh5", "Flt1", "Ramp2", "Sparcl1"
    ],

    "Mast cells": [
        "Cma1", "Mrgprb2", "Ms4a2", "Mcpt4", "Tpsb2", "Fcer1a", "Cpa3", "Hdc"
    ]
}

In [ ]:
sc.tl.dendrogram(adata,'celltype',use_rep="X_pca")
with plt.rc_context({'figure.figsize':(6,5)}):
    sc.pl.dotplot(
        adata,
        groupby="celltype",
        var_names=marker_dict,
        dendrogram=False,
        cmap='RdBu_r', #viridis ,
        #categories_order = order,
        standard_scale="var",  # standard scale: normalize each gene to range from 0 to 1
        save=True
        )

In [ ]:
genes = ["Ptprc","Col1a1","Epcam","Krt8","Krt18","Krt19","Tacstd2"]
df = sc.get.obs_df(adata, keys=genes, use_raw=True)
for g in genes:
    print(g, "pct>0:", (df[g]>0).mean(), "mean:", df[g].mean(), "p99:", np.quantile(df[g], 0.99))

In [ ]:
sc.pl.embedding(
    adata, 
    basis='X_umap_pca',
    color=["Ptprc", "Pecam1", "Dcn","Hmga2"],
    cmap='RdBu_r',
    ncols=4,
    vmax=3,vmin=0,
    show=True,frameon=False,legend_loc=None,
    save=False
)

### 4.11 Final refinement

In [ ]:
adata

In [ ]:
#adata.obs['celltype'] = adata.obs['celltype'].replace('Folr2+ TAMs', 'Ccl8+ TAMs')

In [ ]:
sc.pl.embedding(adata, basis='X_umap_pca',
                color=['celltype'],
                title = ['All cells'],
                #ncols=2, wspace=0.45,
                #legend_loc="on data",
                legend_fontsize=9,
                size=8,
                palette=celltype_colors,
                show=True, 
                frameon=False,
                save='UMAP_anno_all.pdf')

In [ ]:
# ----------------------------
# Config
# ----------------------------
outdir = "./figures"
os.makedirs(outdir, exist_ok=True)

UMAP_BASIS = "X_umap_pca"   # 你已有这个
SCORE_COL = "cnv_score"
STATUS_COL = "predicted_class"  # CNV status column
GROUP_COL = "group"
CELLTYPE_COL = "celltype"
SAMPLE_COL = "sampleID"         # 如果你要 sample-level 统计（推荐）
treatment_order =["PBS","R301"]
# Optional: set order if you have it
GROUP_ORDER = treatment_order if "treatment_order" in globals() else None

# ----------------------------
# Guardrails
# ----------------------------
missing = [c for c in [SCORE_COL, STATUS_COL, GROUP_COL, CELLTYPE_COL] if c not in adata.obs.columns]
if missing:
    raise KeyError(f"Missing columns in adata.obs: {missing}")

if SCORE_COL not in adata.obs.columns or adata.obs[SCORE_COL].isna().all():
    raise ValueError(f"{SCORE_COL} not found or all NA in adata.obs")

# DataFrame for seaborn
df = adata.obs[[SCORE_COL, STATUS_COL, GROUP_COL, CELLTYPE_COL] + ([SAMPLE_COL] if SAMPLE_COL in adata.obs.columns else [])].copy()

# keep only finite numeric scores
df[SCORE_COL] = pd.to_numeric(df[SCORE_COL], errors="coerce")
df = df.loc[np.isfinite(df[SCORE_COL])].copy()

# apply categorical order for group if provided
if GROUP_ORDER is not None:
    df[GROUP_COL] = pd.Categorical(df[GROUP_COL], categories=GROUP_ORDER, ordered=True)

# ----------------------------
# 1) UMAP overview: cnv_score + predicted_class
# ----------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

sc.pl.embedding(
    adata,
    basis=UMAP_BASIS,
    color=SCORE_COL,
    ax=axes[0],
    show=False,
    title="CNV Score (CopyKAT-like pipeline)",
    cmap="RdBu_r",
    vmin=0, vmax=2
)

sc.pl.embedding(
    adata,
    basis=UMAP_BASIS,
    color=STATUS_COL,
    ax=axes[1],
    show=False,
    title="CNV Analysis Status"
)

fig.savefig(os.path.join(outdir, "cnv_umap_overview.pdf"), dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

# ----------------------------
# 2) Boxplot: CNV scores by treatment group (cell-level)
# ----------------------------
fig, ax = plt.subplots(figsize=(10, 6))

sns.boxplot(
    data=df,
    x=GROUP_COL,
    y=SCORE_COL,
    order=GROUP_ORDER,
    palette=color if "group_colors" in globals() else None,
    ax=ax
)
# optional: add points (downsample if huge)
# sns.stripplot(data=df.sample(min(5000, len(df)), random_state=0),
#               x=GROUP_COL, y=SCORE_COL, order=GROUP_ORDER,
#               color="black", size=1.5, alpha=0.25, ax=ax)

ax.set_title("CNV Score by Treatment Group (cell-level)")
ax.set_xlabel("")
ax.set_ylabel("CNV Score")
ax.tick_params(axis="x", rotation=45)
for lab in ax.get_xticklabels():
    lab.set_ha("right")

fig.tight_layout()
fig.savefig(os.path.join(outdir, "cnv_by_group_celllevel.pdf"), dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

# ----------------------------
# 3) Boxplot: CNV scores by celltype (cell-level)
# ----------------------------
fig, ax = plt.subplots(figsize=(12, 6))

sns.boxplot(
    data=df,
    x=CELLTYPE_COL,
    y=SCORE_COL,
    palette=celltype_colors if "celltype_colors" in globals() else None,
    ax=ax
)

ax.set_title("CNV Score by Cell Type (cell-level)")
ax.set_xlabel("")
ax.set_ylabel("CNV Score")
ax.tick_params(axis="x", rotation=45)
for lab in ax.get_xticklabels():
    lab.set_ha("right")

fig.tight_layout()
fig.savefig(os.path.join(outdir, "cnv_by_celltype_celllevel.pdf"), dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

# ----------------------------
# 4) (推荐) Sample-level summary to avoid pseudo-replication
#     每个 sample 聚合成一个值，再做箱线图
# ----------------------------
if SAMPLE_COL in df.columns:
    # per-sample mean (you can switch to median)
    df_s = (df.groupby([GROUP_COL, SAMPLE_COL], observed=True)[SCORE_COL]
              .mean()
              .reset_index(name="cnv_score_mean"))

    # keep order
    if GROUP_ORDER is not None:
        df_s[GROUP_COL] = pd.Categorical(df_s[GROUP_COL], categories=GROUP_ORDER, ordered=True)

    fig, ax = plt.subplots(figsize=(10, 6))

    sns.boxplot(
        data=df_s,
        x=GROUP_COL,
        y="cnv_score_mean",
        order=GROUP_ORDER,
        palette=color if "group_colors" in globals() else None,
        ax=ax
    )
    sns.stripplot(
        data=df_s,
        x=GROUP_COL,
        y="cnv_score_mean",
        order=GROUP_ORDER,
        color="black",
        size=4,
        jitter=0.18,
        alpha=0.75,
        ax=ax
    )

    ax.set_title("CNV Score by Treatment Group (sample-level mean)")
    ax.set_xlabel("")
    ax.set_ylabel("Mean CNV Score (per sample)")
    ax.tick_params(axis="x", rotation=45)
    for lab in ax.get_xticklabels():
        lab.set_ha("right")

    fig.tight_layout()
    fig.savefig(os.path.join(outdir, "cnv_by_group_samplelevel_mean.pdf"), dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

### 4.12 UMAP Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes = axes.flatten()

for idx, (ax, group) in enumerate(zip(axes, treatment_order)):
    sc.pl.embedding(
        adata[adata.obs['group'] == group],
        basis='X_umap_pca',
        color=["celltype"],
        title=group,
        size=16,
        palette=celltype_colors,
        show=False,
        frameon=False,
        ax=ax,
        legend_loc='on data' if idx == 0 else None,
        legend_fontsize=8,
        legend_fontoutline=2
    )

plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, 'UMAP_2groups_comparison.pdf'),
            bbox_inches='tight', dpi=300)
plt.show()
print(f"Saved to: {os.path.join(RESULTS_DIR, 'UMAP_groups_comparison.pdf')}")

### 4.13 Distribution

In [ ]:
ctab = pd.crosstab(adata.obs['celltype'],adata.obs['group'],margins=True)
print(ctab)

### 4.14 Save

In [ ]:
adata.write(DATA_DIR+ '/Step3-sce_annotated.h5ad', compression='gzip')

### 4.15 Export Metadata

In [ ]:
meta = pd.DataFrame(data=adata.obs)
meta.to_csv(RESULTS_DIR + '/sce_metadata_all.csv',sep="\t") 

---

## 5. Proportion Analysis

### 5.1 Load Data

In [ ]:
adata = sc.read(DATA_DIR+ '/Step3-sce_annotated.h5ad')
adata

### 5.2 Visualize

In [ ]:
sc.pl.embedding(adata, basis='X_umap_pca',
                color=['celltype'],
                title = ['All cells'],
                legend_loc="on data",
                legend_fontsize=5,
                size=15,
                palette=celltype_colors,
                show=True, frameon=False,
                save=True)

### 5.3 Group Counts

In [ ]:
adata.obs['group'].value_counts()

### 5.5 Cell Count Barplot

In [ ]:
# ==============================================================================
# Cell Count Barplot by Group (5 Treatment Groups)
# ==============================================================================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Output directory
prop_fig_dir = os.path.join(RESULTS_DIR, 'figures')
os.makedirs(prop_fig_dir, exist_ok=True)

# Get cell type order by total count
celltype_order = adata.obs['celltype'].value_counts().index.tolist()

# Count cells per group and celltype
count_df = adata.obs.groupby(['group', 'celltype']).size().reset_index(name='count')

# Filter to existing groups
existing_groups = [g for g in treatment_order if g in count_df['group'].values]

# Plot
fig, ax = plt.subplots(figsize=(16, 7))
sns.barplot(
    data=count_df,
    x='celltype',
    y='count',
    order=celltype_order,
    hue='group',
    hue_order=existing_groups,
    palette=color,
    ax=ax
)

# Add count labels on bars
for patch in ax.patches:
    height = patch.get_height()
    if height > 0 and not pd.isna(height):
        ax.text(
            x=patch.get_x() + patch.get_width() / 2,
            y=height,
            s=f'{int(height)}',
            ha='center',
            va='bottom',
            fontsize=8,
            rotation=90
        )

ax.set_xlabel("")
ax.set_title('Cell Counts per Cell Type by Treatment Group', fontsize=14)
ax.set_ylabel('Cell count', fontsize=12)
plt.xticks(rotation=45, ha='right')
ax.legend(title='Group', bbox_to_anchor=(1.02, 1), loc='upper left')

plt.tight_layout()
plt.savefig(
    os.path.join(prop_fig_dir, 'Fig_Barplot_cell_counts_by_group.pdf'),
    bbox_inches='tight',
    dpi=300
)
plt.show()
plt.close()

print(f"Saved: {os.path.join(prop_fig_dir, 'Fig_Barplot_cell_counts_by_group.pdf')}")


### 5.6 Stacked Barplot - Proportion

In [ ]:
# ==============================================================================
# Stacked Barplot - Cell Proportion BY GROUP (3 Treatment Groups)
# ==============================================================================

import pandas as pd
import matplotlib.pyplot as plt
import os

# Output directory
prop_fig_dir = os.path.join(RESULTS_DIR, 'figures')
os.makedirs(prop_fig_dir, exist_ok=True)

# Calculate proportions per sample
counts_df = pd.crosstab(adata.obs['sampleID'], adata.obs['celltype'])
proportions_df = counts_df.div(counts_df.sum(axis=1), axis=0)

# Add group info
sample_to_group = adata.obs[['sampleID', 'group']].drop_duplicates().set_index('sampleID')
proportions_df = proportions_df.join(sample_to_group)

# Calculate GROUP means (not sample-level)
group_props = proportions_df.groupby('group')[proportions_df.columns[:-1]].mean()

# Reindex to treatment order
existing_groups = [g for g in treatment_order if g in group_props.index]
group_props = group_props.reindex(existing_groups)

# Define celltype colors
if 'celltype_colors' not in globals():
    import matplotlib.cm as cm
    celltypes = group_props.columns
    colors = cm.tab20.colors[:len(celltypes)]
    celltype_colors_local = dict(zip(celltypes, colors))
else:
    celltype_colors_local = celltype_colors

plot_colors = [celltype_colors_local.get(ct, 'gray') for ct in group_props.columns]

# Plot stacked barplot BY GROUP
fig, ax = plt.subplots(figsize=(8, 7))
group_props.plot(
    kind='bar',
    stacked=True,
    color=plot_colors,
    ax=ax,
    edgecolor='white',
    linewidth=0.5
)

ax.set_ylabel('Cell Proportion', fontsize=12)
ax.set_xlabel('Treatment Group', fontsize=12)
ax.set_title('Cell Type Composition by Treatment Group', fontsize=14)
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), title='Cell Type', fontsize=9)

# Rotate x labels
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.savefig(
    os.path.join(prop_fig_dir, 'Fig_StackedBar_cell_proportion_BY_GROUP.pdf'),
    dpi=300,
    bbox_inches='tight'
)
plt.show()
plt.close()

print(f"Saved: {os.path.join(prop_fig_dir, 'Fig_StackedBar_cell_proportion_BY_GROUP.pdf')}")

# Print the data for reference
print("\nGroup proportions:")
print(group_props.round(4))


### 5.7 Stacked Barplot - TME

In [ ]:
# ==============================================================================
# Stacked Barplot - Malignant Cell State Composition BY GROUP
# ==============================================================================

import pandas as pd
import matplotlib.pyplot as plt
import os

prop_fig_dir = os.path.join(RESULTS_DIR, 'figures')
os.makedirs(prop_fig_dir, exist_ok=True)

# ------------------------------------------------------------------------------
# Define malignant cell states
# ------------------------------------------------------------------------------

malignant_celltypes = [
    "Mesenchymal-like malignant cells",
    "Growth factor-high malignant cells",
    "Translation-high malignant cells",
    "Hypoxic/glycolytic malignant cells",
    "Cycling malignant cells",
    "IFN-stimulated malignant cells",
    "Proliferating malignant cells",
    "CDH13+ mesenchymal-like malignant cells"
]

# Keep only malignant cell types that actually exist in adata.obs['celltype']
existing_malignant_celltypes = [
    ct for ct in malignant_celltypes
    if ct in adata.obs["celltype"].unique()
]

if len(existing_malignant_celltypes) == 0:
    raise ValueError("No malignant cell types found in adata.obs['celltype']. Please check celltype names.")

print("Malignant cell types used:")
print(existing_malignant_celltypes)

# ------------------------------------------------------------------------------
# Calculate malignant cell state proportions per sample
# ------------------------------------------------------------------------------

# Count cells by sample and cell type
counts_df = pd.crosstab(adata.obs["sampleID"], adata.obs["celltype"])

# Select malignant cell states only
counts_malignant = counts_df.reindex(columns=existing_malignant_celltypes, fill_value=0)

# Normalize within malignant compartment per sample
# Each sample sums to 1 across malignant states
sample_totals = counts_malignant.sum(axis=1)

# Avoid division by zero if a sample has no malignant cells
proportions_df = counts_malignant.div(sample_totals.replace(0, pd.NA), axis=0)

# Add group information
sample_to_group = (
    adata.obs[["sampleID", "group"]]
    .drop_duplicates()
    .set_index("sampleID")
)

proportions_df = proportions_df.join(sample_to_group)

# Remove samples without malignant cells, if any
proportions_df = proportions_df.dropna(subset=existing_malignant_celltypes, how="all")

# ------------------------------------------------------------------------------
# Calculate group means
# ------------------------------------------------------------------------------

group_props = (
    proportions_df
    .groupby("group")[existing_malignant_celltypes]
    .mean()
)

# Reindex treatment groups
existing_groups = [g for g in treatment_order if g in group_props.index]
group_props = group_props.reindex(existing_groups)

# ------------------------------------------------------------------------------
# Define colors
# ------------------------------------------------------------------------------

if "celltype_colors" not in globals():
    import matplotlib.cm as cm
    colors = cm.tab20.colors[:len(existing_malignant_celltypes)]
    celltype_colors_local = dict(zip(existing_malignant_celltypes, colors))
else:
    celltype_colors_local = celltype_colors

plot_colors = [
    celltype_colors_local.get(ct, "gray")
    for ct in group_props.columns
]

# ------------------------------------------------------------------------------
# Plot malignant-only composition by group
# ------------------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(8, 7))

group_props.plot(
    kind="bar",
    stacked=True,
    color=plot_colors,
    ax=ax,
    edgecolor="white",
    linewidth=0.5
)

ax.set_ylabel("Proportion within malignant cells", fontsize=12)
ax.set_xlabel("Treatment Group", fontsize=12)
ax.set_title(
    "Malignant Cell State Composition by Group",
    fontsize=14,
    fontweight="bold"
)

ax.legend(
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    title="Malignant Cell State",
    fontsize=9
)

plt.xticks(rotation=45, ha="right")
plt.tight_layout()

out_file = os.path.join(
    prop_fig_dir,
    "Fig_StackedBar_MalignantStates_BY_GROUP.pdf"
)

plt.savefig(
    out_file,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

print(f"Saved: {out_file}")

print("\nMalignant cell state proportions by group:")
print(group_props.round(4))

### 5.8 Alluvial Plot

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import os

def plot_alluvial_single_ax(group_props, celltypes, colors_dict,
                            title, outpath=None, figsize=(12, 7),
                            bar_width=0.35, band_alpha=0.28):

    groups = list(group_props.index)
    n = len(groups)
    x = np.arange(n)

    fig, ax = plt.subplots(1, 1, figsize=figsize)

    # 计算每个 group 的堆叠起点 bottom
    bottoms = {g: 0.0 for g in groups}

    # 为了能画“带子”，我们需要存每个 celltype 在每个 group 的 (y0,y1)
    yspans = {g: {} for g in groups}

    # 先画所有堆叠柱，并记录区间
    for ct in celltypes:
        for i, g in enumerate(groups):
            h = float(group_props.loc[g, ct]) if ct in group_props.columns else 0.0
            y0 = bottoms[g]
            y1 = y0 + h
            yspans[g][ct] = (y0, y1)

            ax.bar(x[i], h, bottom=y0, width=bar_width,
                   color=colors_dict.get(ct, "gray"),
                   edgecolor="white", linewidth=0.6)

            bottoms[g] = y1

    # 再画相邻组之间的连接带（全部画在同一个 ax 上）
    for i in range(n - 1):
        g1, g2 = groups[i], groups[i + 1]
        x_left = x[i] + bar_width / 2
        x_right = x[i + 1] - bar_width / 2

        for ct in celltypes:
            y0_l, y1_l = yspans[g1][ct]
            y0_r, y1_r = yspans[g2][ct]
            if (y1_l - y0_l) <= 0 and (y1_r - y0_r) <= 0:
                continue

            verts = [
                (x_left,  y0_l),
                (x_right, y0_r),
                (x_right, y1_r),
                (x_left,  y1_l),
            ]
            poly = patches.Polygon(
                verts, closed=True,
                facecolor=colors_dict.get(ct, "gray"),
                edgecolor="none",
                alpha=band_alpha
            )
            ax.add_patch(poly)

    ax.set_xticks(x)
    ax.set_xticklabels(groups, rotation=45, ha="right")
    ax.set_xlim(-0.6, n - 1 + 0.6)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Proportion")
    ax.set_title(title)

    # legend（单独用patch做，避免bar重复太多）
    handles = [patches.Patch(color=colors_dict.get(ct, "gray"), label=ct) for ct in celltypes]
    ax.legend(handles=handles, title="Cell Type",
              loc="center left", bbox_to_anchor=(1.02, 0.5),
              fontsize=8, title_fontsize=9, frameon=False)

    plt.tight_layout()
    if outpath:
        fig.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

In [ ]:
print("Generating Alluvial Plot (BY GROUP) - ALL cell types...")

# Calculate proportions per sample then average by group
counts_df = pd.crosstab(adata.obs['sampleID'], adata.obs['celltype'])
proportions_df = counts_df.div(counts_df.sum(axis=1), axis=0)

# Add group and calculate GROUP means
sample_to_group = adata.obs[['sampleID', 'group']].drop_duplicates().set_index('sampleID')
proportions_df = proportions_df.join(sample_to_group)

celltype_list_all = celltype_order
celltype_list_all = [ct for ct in celltype_list_all if ct in proportions_df.columns]

# Calculate GROUP-level proportions
group_props_all = proportions_df.groupby('group')[celltype_list_all].mean()
group_props_all = group_props_all.reindex([g for g in treatment_order 
                                           if g in group_props_all.index])

# 用你现成的 group_props_all / celltype_list_all / celltype_colors_local
outpath = os.path.join(prop_fig_dir, "Fig_Alluvial_BY_GROUP_ALL_cells.pdf")
plot_alluvial_single_ax(
    group_props=group_props_all,
    celltypes=celltype_list_all,
    colors_dict=celltype_colors_local,
    title="Cell Type Proportion Shifts by Group (ALL Cells)",
    outpath=outpath,
    figsize=(8, 6)
)
print(f"Saved: {outpath}")

In [ ]:
print("\nGenerating Alluvial Plot (BY GROUP) - TME only (no tumor)...")

# Exclude tumor and recalculate
counts_no_tumor = counts_df.drop(columns=['Malignant cells'], errors='ignore')
proportions_tme = counts_no_tumor.div(counts_no_tumor.sum(axis=1), axis=0)
proportions_tme = proportions_tme.join(sample_to_group)

celltype_list_tme = ['Monocytes', 'Mono-to-macro transitional cells',
    'Resident macrophages', 'Activated macrophages','ISG+ myeloid cells','Proliferating myeloid cells',
    'cDCs','pDCs',
    'Activated T cells','Exhausted-like T cells','NK cells',
    'Fibroblasts', 'Endothelial cells',]
celltype_list_tme = [ct for ct in celltype_list_tme if ct in proportions_tme.columns]

# Calculate GROUP-level TME proportions
group_props_tme = proportions_tme.groupby('group')[celltype_list_tme].mean()
group_props_tme = group_props_tme.reindex([g for g in treatment_order 
                                          if g in group_props_tme.index])
outpath = os.path.join(prop_fig_dir, "Fig_Alluvial_BY_GROUP_TME_no_tumor.pdf")
plot_alluvial_single_ax(
    group_props=group_props_tme,
    celltypes=celltype_list_tme,
    colors_dict=celltype_colors_local,
    title="Cell Type Proportion Shifts by Group (TME, Tumor Excluded)",
    outpath=outpath,
    figsize=(8, 6)
)
print(f"Saved: {outpath}")

### 5.9 ANOVA

In [ ]:
# ==============================================================================
# 5-Group Differential Cell Proportion Analysis (sample-level)
# 1) Build sample-level celltype proportions
# 2) Global tests per cell type:
#    - ANOVA on arcsin(sqrt(p))
#    - Kruskal–Wallis on raw p
#    - BH-FDR across cell types
# 3) Post-hoc:
#    - Tukey HSD on transformed values (within-celltype adjusted p)
#    - Pairwise MWU on raw p + within-celltype correction (FDR or Bonferroni)
# 4) Per-celltype boxplot saved individually with correct group colors + Tukey stars (*,**,***)
# ==============================================================================

import os
import itertools
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from scipy.stats import f_oneway, kruskal, mannwhitneyu
import statsmodels.stats.multitest as smm
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# -----------------------------
# User settings (EDIT HERE)
# -----------------------------
prop_fig_dir = os.path.join(RESULTS_DIR, "al_proportion")
os.makedirs(prop_fig_dir, exist_ok=True)

treatment_groups = [
    'PBS','R301'
]

celltype_list = celltype_order

# IMPORTANT: keys must exactly match adata.obs["group"]
MIN_CELLS_PER_SAMPLE = 200         # set 0 to disable filtering
POSTHOC_CORR = "fdr_bh"            # for MWU pairs within-celltype: "fdr_bh" or "bonferroni"
RUN_POSTHOC_ONLY_IF_GLOBAL_Q_LT = None   # e.g. 0.10 or 0.05; None=always run

# Plot options
MAKE_PLOTS = True
PLOT_FMT = "pdf"                   # "pdf" or "png"
MAX_PAIRS_TO_ANNOTATE = 6          # per plot (avoid clutter)
ANNOTATE_ONLY_SIG = True           # only show p<0.05 stars
PERCENT_YAXIS = False              # show as percent
UNIFY_YLIM = False                 # same y-limit across celltypes

# -----------------------------
# Helpers
# -----------------------------
def arcsin_sqrt_transform(p):
    p = np.clip(np.asarray(p, dtype=float), 0.0, 1.0)
    return np.arcsin(np.sqrt(p))

def fdr_series(pvals: pd.Series, method="fdr_bh") -> pd.Series:
    mask = pvals.notna()
    out = pd.Series(np.nan, index=pvals.index, dtype=float)
    if mask.sum() == 0:
        return out
    _, q, _, _ = smm.multipletests(pvals[mask].values, method=method)
    out.loc[mask] = q
    return out

def safe_global_anova(group_arrays):
    all_vals = np.concatenate([np.asarray(a) for a in group_arrays if len(a) > 0])
    if len(all_vals) == 0 or np.allclose(all_vals, all_vals[0]):
        return np.nan, np.nan
    return f_oneway(*group_arrays)

def safe_global_kw(group_arrays):
    all_vals = np.concatenate([np.asarray(a) for a in group_arrays if len(a) > 0])
    if len(all_vals) == 0 or np.allclose(all_vals, all_vals[0]):
        return np.nan, np.nan
    return kruskal(*group_arrays)

def pairwise_mwu(values, groups, group_order, alternative="two-sided"):
    df = pd.DataFrame({"value": values, "group": groups})
    out_rows = []
    for g1, g2 in itertools.combinations(group_order, 2):
        v1 = df.loc[df["group"] == g1, "value"].astype(float).values
        v2 = df.loc[df["group"] == g2, "value"].astype(float).values

        if len(v1) == 0 or len(v2) == 0:
            out_rows.append({"group1": g1, "group2": g2, "n1": len(v1), "n2": len(v2),
                             "u_stat": np.nan, "p_value": np.nan})
            continue

        try:
            u, p = mannwhitneyu(v1, v2, alternative=alternative)
        except ValueError:
            u, p = np.nan, np.nan

        out_rows.append({"group1": g1, "group2": g2, "n1": len(v1), "n2": len(v2),
                         "u_stat": u, "p_value": p})
    return pd.DataFrame(out_rows)

def p_to_stars(p: float) -> str:
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""

def add_sig_bracket(ax, x1, x2, y, h, text, lw=1.2, fontsize=12):
    ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], c="black", lw=lw, clip_on=False)
    ax.text((x1 + x2) / 2, y + h, text, ha="center", va="bottom",
            fontsize=fontsize, color="black", clip_on=False)

# -----------------------------
# 1) Build sample-level proportions
# -----------------------------
counts_df = pd.crosstab(adata.obs["sampleID"], adata.obs["celltype"])

sample_to_group = (
    adata.obs[["sampleID", "group"]]
    .drop_duplicates()
    .set_index("sampleID")
)

missing_group = counts_df.index.difference(sample_to_group.index)
if len(missing_group) > 0:
    raise ValueError(f"Some sampleIDs lack group annotation: {list(missing_group)[:10]}")

proportions_df = counts_df.div(counts_df.sum(axis=1), axis=0).join(sample_to_group)
proportions_df["n_cells"] = counts_df.sum(axis=1)

# Keep only requested celltypes that exist
celltype_list = [ct for ct in celltype_list if ct in proportions_df.columns]
if len(celltype_list) == 0:
    raise ValueError("None of the requested cell types are present in adata.obs['celltype'].")

# Filter tiny samples (optional)
if MIN_CELLS_PER_SAMPLE and MIN_CELLS_PER_SAMPLE > 0:
    before = proportions_df.shape[0]
    proportions_df = proportions_df.loc[proportions_df["n_cells"] >= MIN_CELLS_PER_SAMPLE].copy()
    after = proportions_df.shape[0]
    print(f"Filtered samples by n_cells >= {MIN_CELLS_PER_SAMPLE}: {before} -> {after}")

# Group order (existing only, in desired order)
existing_groups = [g for g in treatment_groups if g in proportions_df["group"].unique()]
if len(existing_groups) < 2:
    raise ValueError("Fewer than 2 groups present after filtering; cannot compare.")

# Quick QC
print("\nQC: n_samples per group")
print(
    proportions_df.reset_index()
    .groupby("group")["sampleID"]
    .nunique()
    .reindex(existing_groups)
    .to_string()
)
print("\nQC: cells per sample summary")
print(proportions_df["n_cells"].describe().to_string())

# Ensure colors cover existing groups
missing_colors = [g for g in existing_groups if g not in color]
if missing_colors:
    raise ValueError(f"group_colors missing keys for groups: {missing_colors}")

# -----------------------------
# 2) Global tests per cell type (sample-level)
# -----------------------------
global_rows = []
for ct in celltype_list:
    raw_arrays, tr_arrays, ns = [], [], []
    for g in existing_groups:
        vals = proportions_df.loc[proportions_df["group"] == g, ct].astype(float).values
        raw_arrays.append(vals)
        tr_arrays.append(arcsin_sqrt_transform(vals))
        ns.append(len(vals))

    # require >=2 samples per group for ANOVA variance estimate
    if any(n < 2 for n in ns):
        global_rows.append({
            "cell_type": ct,
            "group_ns": ",".join(map(str, ns)),
            "anova_f": np.nan, "anova_p": np.nan,
            "kw_h": np.nan, "kw_p": np.nan,
            "note": "Skipped: some group has <2 samples"
        })
        continue

    f_stat, p_anova = safe_global_anova(tr_arrays)
    h_stat, p_kw = safe_global_kw(raw_arrays)

    global_rows.append({
        "cell_type": ct,
        "group_ns": ",".join(map(str, ns)),
        "anova_f": f_stat, "anova_p": p_anova,
        "kw_h": h_stat, "kw_p": p_kw,
        "note": ""
    })

global_df = pd.DataFrame(global_rows)
global_df["anova_q_fdr"] = fdr_series(global_df["anova_p"], method="fdr_bh")
global_df["kw_q_fdr"] = fdr_series(global_df["kw_p"], method="fdr_bh")

# Add group mean/sem (raw proportions)
group_means = proportions_df.groupby("group")[celltype_list].mean().reindex(existing_groups)
group_sems  = proportions_df.groupby("group")[celltype_list].sem().reindex(existing_groups)
for g in existing_groups:
    global_df[f"mean_{g}"] = global_df["cell_type"].map(group_means.loc[g].to_dict())
    global_df[f"sem_{g}"]  = global_df["cell_type"].map(group_sems.loc[g].to_dict())

global_df["min_q"] = np.nanmin(
    np.vstack([global_df["anova_q_fdr"].values, global_df["kw_q_fdr"].values]),
    axis=0
)
global_df = global_df.sort_values(["min_q", "cell_type"], ascending=[True, True]).reset_index(drop=True)

global_out = os.path.join(prop_fig_dir, "CellProportion_5group_GLOBAL_ANOVA_KW_FDR.csv")
global_df.to_csv(global_out, index=False)
print(f"\nSaved global results: {global_out}")

print("\nTop global results (by min_q):")
print(global_df[["cell_type", "anova_p", "anova_q_fdr", "kw_p", "kw_q_fdr", "group_ns"]].head(20).to_string(index=False))

# -----------------------------
# 3) Post-hoc A: Tukey HSD (ANOVA follow-up; already adjusted within-celltype)
# -----------------------------
tukey_rows = []
for ct in celltype_list:
    g_row = global_df.loc[global_df["cell_type"] == ct].iloc[0]

    if RUN_POSTHOC_ONLY_IF_GLOBAL_Q_LT is not None:
        if not (pd.notna(g_row["anova_q_fdr"]) and g_row["anova_q_fdr"] < RUN_POSTHOC_ONLY_IF_GLOBAL_Q_LT):
            continue

    sub = proportions_df.loc[proportions_df["group"].isin(existing_groups), ["group", ct]].dropna().copy()
    if sub.empty or sub["group"].nunique() < 2:
        continue

    sub["value_tr"] = arcsin_sqrt_transform(sub[ct].astype(float).values)

    # If no variability, skip
    if np.allclose(sub["value_tr"].values, sub["value_tr"].values[0]):
        continue

    try:
        tuk = pairwise_tukeyhsd(endog=sub["value_tr"].values, groups=sub["group"].values, alpha=0.05)
        tdf = pd.DataFrame(tuk.summary().data[1:], columns=tuk.summary().data[0])
        for _, r in tdf.iterrows():
            tukey_rows.append({
                "cell_type": ct,
                "group1": r["group1"],
                "group2": r["group2"],
                "meandiff_tr": float(r["meandiff"]),
                "p_adj_tukey": float(r["p-adj"]),
                "lower_tr": float(r["lower"]),
                "upper_tr": float(r["upper"]),
                "reject_0p05": bool(r["reject"]),
                "global_anova_p": g_row["anova_p"],
                "global_anova_q_fdr": g_row["anova_q_fdr"],
            })
    except Exception as e:
        tukey_rows.append({
            "cell_type": ct, "group1": None, "group2": None,
            "meandiff_tr": np.nan, "p_adj_tukey": np.nan,
            "lower_tr": np.nan, "upper_tr": np.nan, "reject_0p05": False,
            "global_anova_p": g_row["anova_p"],
            "global_anova_q_fdr": g_row["anova_q_fdr"],
            "error": str(e)
        })

tukey_df = pd.DataFrame(tukey_rows)
tukey_out = os.path.join(prop_fig_dir, "CellProportion_5group_POSTHOC_TukeyHSD_arcsin.csv")
tukey_df.to_csv(tukey_out, index=False)
print(f"Saved Tukey post-hoc: {tukey_out}")

# -----------------------------
# 3) Post-hoc B: Pairwise MWU (KW follow-up) + within-celltype correction
# -----------------------------
mwu_blocks = []
for ct in celltype_list:
    g_row = global_df.loc[global_df["cell_type"] == ct].iloc[0]

    if RUN_POSTHOC_ONLY_IF_GLOBAL_Q_LT is not None:
        if not (pd.notna(g_row["kw_q_fdr"]) and g_row["kw_q_fdr"] < RUN_POSTHOC_ONLY_IF_GLOBAL_Q_LT):
            continue

    sub = proportions_df.loc[proportions_df["group"].isin(existing_groups), ["group", ct]].dropna().copy()
    if sub.empty or sub["group"].nunique() < 2:
        continue

    pw = pairwise_mwu(
        values=sub[ct].astype(float).values,
        groups=sub["group"].values,
        group_order=existing_groups,
        alternative="two-sided"
    )
    pw["p_corr"] = fdr_series(pw["p_value"], method=POSTHOC_CORR).values
    pw["reject_corr_0p05"] = pw["p_corr"] < 0.05
    pw.insert(0, "cell_type", ct)
    pw["global_kw_p"] = g_row["kw_p"]
    pw["global_kw_q_fdr"] = g_row["kw_q_fdr"]
    pw["corr_method_within_celltype"] = POSTHOC_CORR
    mwu_blocks.append(pw)

mwu_df = pd.concat(mwu_blocks, ignore_index=True) if len(mwu_blocks) else pd.DataFrame()
mwu_out = os.path.join(prop_fig_dir, f"CellProportion_5group_POSTHOC_MWU_withinCelltype_{POSTHOC_CORR}.csv")
mwu_df.to_csv(mwu_out, index=False)
print(f"Saved MWU post-hoc: {mwu_out}")

# -----------------------------
# 4) Plot per celltype with improved typography/layout + Tukey star annotations
# -----------------------------
if MAKE_PLOTS:
    plot_dir = os.path.join(prop_fig_dir, "celltype_proportion_plots_annotated")
    os.makedirs(plot_dir, exist_ok=True)

    # ---- Theme / rcParams (publication-ish) ----
    sns.set_theme(style="whitegrid", context="paper")  # 'paper' keeps fonts modest
    plt.rcParams.update({
        "figure.dpi": 150,
        "savefig.dpi": 300,
        "font.family": "DejaVu Sans",   # safe default; switch to Arial if installed
        "axes.titlesize": 14,
        "axes.titleweight": "semibold",
        "axes.labelsize": 12,
        "xtick.labelsize": 10.5,
        "ytick.labelsize": 10.5,
        "legend.fontsize": 10,
        "axes.linewidth": 1.0,
        "grid.linewidth": 0.8,
    })

    # Long format once
    long_df = (
        proportions_df.reset_index()
        .melt(id_vars=["group"], value_vars=celltype_list, var_name="cell_type", value_name="proportion")
        .dropna()
    )
    long_df["group"] = pd.Categorical(long_df["group"], categories=existing_groups, ordered=True)

    global_max = long_df["proportion"].max() if (UNIFY_YLIM and not long_df.empty) else None

    # Pre-index Tukey/global for speed
    tukey_by_ct = {ct: df for ct, df in tukey_df.groupby("cell_type")} if (not tukey_df.empty and "cell_type" in tukey_df.columns) else {}
    global_by_ct = global_df.set_index("cell_type") if ("cell_type" in global_df.columns) else pd.DataFrame()

    def fmt_q(x):
        try:
            x = float(x)
        except Exception:
            return "NA"
        return "NA" if np.isnan(x) else f"{x:.2g}"

    def clean_name(s: str) -> str:
        return (s.replace("/", "_").replace(" ", "_").replace("+", "plus"))

    for ct in celltype_list:
        sub = long_df.loc[long_df["cell_type"] == ct, ["group", "proportion"]].copy()
        if sub.empty:
            continue

        # fetch global q-values from global_df
        if isinstance(global_by_ct, pd.DataFrame) and (not global_by_ct.empty) and (ct in global_by_ct.index):
            anova_q = global_by_ct.loc[ct, "anova_q_fdr"]
            kw_q    = global_by_ct.loc[ct, "kw_q_fdr"]
        else:
            anova_q = np.nan
            kw_q    = np.nan

        # ---- Figure: slightly wider for long labels ----
        fig, ax = plt.subplots(figsize=(7, 4.6), constrained_layout=True)

        # Grid aesthetics
        ax.grid(True, axis="y", color="#000000", alpha=0.10)
        ax.grid(False, axis="x")

        # Boxplot
        sns.boxplot(
            data=sub, x="group", y="proportion",
            order=existing_groups,
            palette=color,
            width=0.62,
            fliersize=0,
            linewidth=1.1,
            ax=ax
        )

        # Points: slightly smaller + white edge to separate from boxes
        sns.stripplot(
            data=sub, x="group", y="proportion",
            order=existing_groups,
            color="black",
            size=4.3,
            jitter=0.15,
            alpha=0.75,
            edgecolor="white",
            linewidth=0.4,
            ax=ax
        )

        # Titles/labels
        ax.set_title(f"{ct}\nANOVA (q) = {fmt_q(anova_q)}   |   Kruskal (q) = {fmt_q(kw_q)}", pad=10)
        ax.set_xlabel("")
        ax.set_ylabel("Proportion (per sample)")

        # x tick labels: less rotation, aligned
        ax.tick_params(axis="x", rotation=28)
        for lab in ax.get_xticklabels():
            lab.set_ha("right")

        # y-limits
        ymax_data = float(sub["proportion"].max()) if np.isfinite(sub["proportion"].max()) else 0.0
        ymax_base = float(global_max) if (global_max is not None and np.isfinite(global_max)) else ymax_data
        ymax = max(0.05, ymax_base * 1.15)
        ax.set_ylim(0, ymax)

        # Percent axis option
        if PERCENT_YAXIS:
            ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0, decimals=0))
            ax.set_ylabel("Proportion (%)")

        # ---- Tukey annotations ----
        tct = tukey_by_ct.get(ct, None)
        if tct is not None and not tct.empty and {"group1", "group2", "p_adj_tukey"}.issubset(tct.columns):
            ann = tct.loc[
                tct["group1"].isin(existing_groups) & tct["group2"].isin(existing_groups),
                ["group1", "group2", "p_adj_tukey"]
            ].copy()

            ann["p_adj_tukey"] = pd.to_numeric(ann["p_adj_tukey"], errors="coerce")
            ann = ann.dropna(subset=["p_adj_tukey"])
            ann["stars"] = ann["p_adj_tukey"].apply(p_to_stars)

            if ANNOTATE_ONLY_SIG:
                ann = ann.loc[ann["stars"] != ""].copy()

            ann = ann.sort_values("p_adj_tukey", ascending=True).head(MAX_PAIRS_TO_ANNOTATE)

            if not ann.empty:
                xmap = {g: i for i, g in enumerate(existing_groups)}
                y_min, y_max = ax.get_ylim()
                y_range = max(1e-6, y_max - y_min)

                # start a bit above top of data; step derived from range so it scales nicely
                y0 = max(ymax_data + 0.06 * y_range, 0.02)
                step = 0.08 * y_range
                h = 0.018 * y_range

                for k, (_, r) in enumerate(ann.iterrows()):
                    g1, g2 = r["group1"], r["group2"]
                    x1, x2 = xmap[g1], xmap[g2]
                    if x1 > x2:
                        x1, x2 = x2, x1
                    y = y0 + k * step
                    add_sig_bracket(ax, x1, x2, y=y, h=h, text=r["stars"], fontsize=12)

                # ensure enough headroom for brackets
                new_top = y0 + (len(ann) - 1) * step + h + 0.06 * y_range
                ax.set_ylim(y_min, max(y_max, new_top))

        sns.despine(ax=ax, left=False, bottom=False)

        safe = clean_name(ct)
        fout = os.path.join(plot_dir, f"Boxplot_{safe}_TukeyStars.{PLOT_FMT}")
        fig.savefig(fout, bbox_inches="tight")
        plt.close(fig)

    print(f"Saved annotated per-celltype plots to: {plot_dir}")

print("\nALL DONE.")

### 5.10 Boxplot

In [ ]:
# ==============================================================================
# Boxplot Visualization - Cell Proportions by Group
# ==============================================================================

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

# Output directory
prop_fig_dir = os.path.join(RESULTS_DIR, 'al_proportion')
os.makedirs(prop_fig_dir, exist_ok=True)

# Calculate proportions
counts_df = pd.crosstab(adata.obs['sampleID'], adata.obs['celltype'])
proportions_df = counts_df.div(counts_df.sum(axis=1), axis=0)

# Add group info
sample_to_group = adata.obs[['sampleID', 'group']].drop_duplicates().set_index('sampleID')
proportions_df = proportions_df.join(sample_to_group)

# Define cell types

celltype_list = [ct for ct in celltype_list if ct in proportions_df.columns]

# Melt for plotting
melted_df = pd.melt(
    proportions_df,
    id_vars=['group'],
    value_vars=celltype_list,
    var_name='Cell Type',
    value_name='Proportion'
)


# Faceted boxplot
g = sns.FacetGrid(
    melted_df, 
    col='Cell Type', 
    col_wrap=4, 
    height=3, 
    aspect=1.2, 
    sharey=False
)

g.map(sns.boxplot, 'group', 'Proportion', palette=color, order=color.keys())
g.map(sns.stripplot, 'group', 'Proportion', color='black', size=5, jitter=True, alpha=0.7)

# Rotate x labels
for ax in g.axes.flat:
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)

g.set_titles(col_template='{col_name}', size=10)
g.set_xlabels('')
g.fig.suptitle('Cell Type Proportions by Treatment Group', fontsize=14, y=1.02)

plt.tight_layout()
plt.savefig(
    os.path.join(prop_fig_dir, 'Fig_Boxplot_cell_proportions_by_group.pdf'),
    dpi=300,
    bbox_inches='tight'
)
plt.show()
plt.close()

print(f"Saved: {os.path.join(prop_fig_dir, 'Fig_Boxplot_cell_proportions_by_group.pdf')}")


---

## 6. DEG Analysis

**Comparisons vs NC (Control):**
1. RC148 vs NC
2. AK112 vs NC

### 6.1 Load Data

In [ ]:
adata = sc.read(DATA_DIR+ '/Step3-sce_annotated.h5ad')
adata

### 6.2 Check Distribution

In [ ]:
adata.raw.shape

In [ ]:
adata.obs['celltype'].value_counts()

### 6.3 Volcano Function

In [ ]:
# Ensure libraries imported
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from adjustText import adjust_text

def plot_volcano_publication_quality(
    degs_df,
    title,
    subtitle,
    top_n_up=15,
    top_n_down=15,
    genes_to_highlight=None,
    lfc_threshold=1.0,
    pval_threshold=0.05,
    palette=None,
    savepath=None,
):
    """
    绘制一幅具有出版质量的火山图。
    
    Parameters:
    - degs_df: DEG DataFrame。
    - title: Main title。
    - subtitle: Subtitle。
    - top_n_up: Top N up genes。
    - top_n_down: Top N down genes。
    - genes_to_highlight: Gene list，这些基因将被特别标记。
    - lfc_threshold: Log2FC threshold。
    - pval_threshold: Adjusted p threshold。
    - palette: Color dict。
    """
    df = degs_df.copy()

    # --- 1. 数据准备 ---
    df['-log10_pvals_adj'] = -np.log10(df['pvals_adj'].astype(float) + 1e-300)
    df['status'] = 'Not significant'
    df.loc[(df['logfoldchanges'] > lfc_threshold) & (df['pvals_adj'] < pval_threshold), 'status'] = 'Up-regulated'
    df.loc[(df['logfoldchanges'] < -lfc_threshold) & (df['pvals_adj'] < pval_threshold), 'status'] = 'Down-regulated'
    
    # 颜色配置
    if palette is None:
        palette = {
            'Up-regulated': '#d62728',  # 更鲜亮的红色
            'Down-regulated': '#1f77b4', # 更鲜亮的蓝色
            'Not significant': '#cccccc'
        }

    # --- 2. 绘图 ---
    plt.style.use('seaborn-v0_8-whitegrid') # 使用一个干净的绘图风格
    fig, ax = plt.subplots(figsize=(12, 12))

    # 分别绘制点，以控制透明度和层级
    ax.scatter(
        df[df['status'] == 'Not significant']['logfoldchanges'],
        df[df['status'] == 'Not significant']['-log10_pvals_adj'],
        s=15, alpha=0.4, c=palette['Not significant'], label='Not significant', ec='none'
    )
    ax.scatter(
        df[df['status'] == 'Up-regulated']['logfoldchanges'],
        df[df['status'] == 'Up-regulated']['-log10_pvals_adj'],
        s=30, alpha=0.8, c=palette['Up-regulated'], label='Up-regulated', ec='none'
    )
    ax.scatter(
        df[df['status'] == 'Down-regulated']['logfoldchanges'],
        df[df['status'] == 'Down-regulated']['-log10_pvals_adj'],
        s=30, alpha=0.8, c=palette['Down-regulated'], label='Down-regulated', ec='none'
    )

    # --- 3. 核心改进：分离式标签选择 ---
    df['ranking_score'] = abs(df['logfoldchanges']) * df['-log10_pvals_adj']
    
    up_genes = df[df['status'] == 'Up-regulated'].sort_values('ranking_score', ascending=False).head(top_n_up)
    down_genes = df[df['status'] == 'Down-regulated'].sort_values('ranking_score', ascending=False).head(top_n_down)
    
    genes_to_label_df = pd.concat([up_genes, down_genes])
    
    # 如果有指定要高亮的基因，也加入标签列表
    if genes_to_highlight:
        highlight_df = df[df['names'].isin(genes_to_highlight)]
        genes_to_label_df = pd.concat([genes_to_label_df, highlight_df]).drop_duplicates(subset=['names'])

    texts = []
    for _, row in genes_to_label_df.iterrows():
        texts.append(ax.text(row['logfoldchanges'], row['-log10_pvals_adj'], row['names'], fontsize=12))

    adjust_text(texts, ax=ax,
                arrowprops=dict(arrowstyle='-', color='grey', lw=0.5),
                force_points=(0.2, 0.5),
                force_text=(0.5, 1.0))

    # --- 4. 添加注释和美化 ---
    # 阈值线
    ax.axhline(y=-np.log10(pval_threshold), color='grey', linestyle='--', linewidth=1)
    ax.axvline(x=lfc_threshold, color='grey', linestyle='--', linewidth=1)
    ax.axvline(x=-lfc_threshold, color='grey', linestyle='--', linewidth=1)
    
    # 统计数量注释
    num_up = (df['status'] == 'Up-regulated').sum()
    num_down = (df['status'] == 'Down-regulated').sum()
    ax.text(0.02, 0.98, f'Up: {num_up}\nDown: {num_down}', transform=ax.transAxes,
            fontsize=12, verticalalignment='top', bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.5, ec='none'))

    # 标题和轴标签
    fig.suptitle(title, fontsize=20, weight='bold')
    ax.set_title(subtitle, fontsize=14, pad=10)
    ax.set_xlabel("Log2 Fold Change", fontsize=14)
    ax.set_ylabel("-log10(Adjusted P-value)", fontsize=14)

    # 图例
    ax.legend(loc='upper right', frameon=False, fontsize=12)
    
    # 移除顶部和右侧的轴线
    sns.despine(ax=ax)
    
    plt.tight_layout(rect=[0, 0, 1, 0.96]) # 为主标题留出空间
    if savepath:
        plt.savefig(savepath, dpi=300)
    plt.show()

### 6.4 Run All Comparisons

In [ ]:
# ============================================================
# Full DEG + Volcano + Enrichr(ORA) + Enrichment dotplot pipeline
# ============================================================

# ---- Imports ----
import os
import re
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import gseapy as gp

# ------------------------------------------------------------
# 0) Utilities
# ------------------------------------------------------------
def safe_filename(s: str) -> str:
    return re.sub(r'[^0-9a-zA-Z\-\_\.]', '_', str(s))

def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)
    return path

# ------------------------------------------------------------
# 1) Enrichr dotplot
# ------------------------------------------------------------
def enrichr_dotplot(enr_results: pd.DataFrame, title: str, top_n: int = 15, savepath: str | None = None):
    """
    Plot Enrichr ORA results (gseapy.enrichr(...).results) as dotplot.

    Expected columns (typical Enrichr):
      - 'Term'
      - 'Adjusted P-value'
      - 'Combined Score'
      - 'Overlap' (e.g., "12/200")

    This function is defensive across minor column name differences.
    """
    if enr_results is None or not isinstance(enr_results, pd.DataFrame) or enr_results.empty:
        return None

    df = enr_results.copy()

    # Find columns robustly
    term_col = 'Term' if 'Term' in df.columns else None
    if term_col is None:
        # fallback: first column that looks like term/name
        for c in df.columns:
            if c.lower() in ('term', 'pathway', 'gene_set', 'name'):
                term_col = c
                break
    if term_col is None:
        term_col = df.columns[0]

    q_col = 'Adjusted P-value' if 'Adjusted P-value' in df.columns else None
    if q_col is None:
        for c in df.columns:
            lc = c.lower()
            if ('adjust' in lc or 'adj' in lc) and ('p' in lc):
                q_col = c
                break
    if q_col is None:
        raise ValueError(f"Cannot find adjusted p-value column in Enrichr results: {df.columns.tolist()}")

    score_col = 'Combined Score' if 'Combined Score' in df.columns else None
    if score_col is None:
        for c in df.columns:
            if 'score' in c.lower():
                score_col = c
                break

    overlap_col = 'Overlap' if 'Overlap' in df.columns else None

    # Prep plot table
    df[q_col] = df[q_col].astype(float)
    df['-log10(q)'] = -np.log10(df[q_col] + 1e-300)

    # hits from overlap
    if overlap_col is not None:
        try:
            hits = df[overlap_col].astype(str).str.split('/', expand=True)[0].astype(float)
            df['Hits'] = hits
        except Exception:
            df['Hits'] = 1.0
    else:
        df['Hits'] = 1.0

    df = df.sort_values(q_col, ascending=True).head(top_n).copy()
    df = df.iloc[::-1]  # reverse for top term on top

    # Plot
    plt.figure(figsize=(10, max(4, 0.38 * len(df) + 1.7)))
    ax = plt.gca()

    cvals = df[score_col] if score_col is not None else df['-log10(q)']
    sca = ax.scatter(
        df['-log10(q)'],
        df[term_col],
        s=40 + 12 * df['Hits'],
        c=cvals,
        cmap='viridis',
        alpha=0.92,
        edgecolor='none'
    )

    ax.set_title(title, fontsize=14, weight='bold')
    ax.set_xlabel('-log10(Adjusted P-value)')
    ax.set_ylabel('')

    cbar = plt.colorbar(sca, ax=ax, pad=0.02)
    cbar.set_label(score_col if score_col is not None else '-log10(q)')

    sns.despine(ax=ax, left=False, bottom=False)
    plt.tight_layout()

    if savepath:
        plt.savefig(savepath, dpi=300)
        plt.close()
        return savepath
    else:
        plt.show()
        return None

# ------------------------------------------------------------
# 2) Core runner
# ------------------------------------------------------------
def run_deg_volcano_enrichr_all(
    adata,
    comparisons,
    main_type_col='celltype',
    compare_col='group',
    lfc_threshold=0.5,
    pval_threshold=0.05,
    gene_sets=('GO_Biological_Process_2023',),
    organism='mouse',
    outdir='./DEG_Enrichr_all_comparisons',
    min_cells=20,
    top_n_terms=15,
):
    """
    For each comparison (group1 vs group2) and each celltype:
      1) DEG via scanpy rank_genes_groups (wilcoxon)
      2) Save DEG table
      3) Volcano plot (requires plot_volcano_publication_quality defined)
      4) Enrichr ORA on significant Up and Down lists
      5) Save Enrichr tables + dotplots

    Returns:
      all_results: nested dict [comp_name][celltype] -> dict with degs/sig/enrichr results (DataFrames)
    """
    ensure_dir(outdir)

    all_celltypes = pd.Index(adata.obs[main_type_col].unique()).tolist()
    all_results = {}

    # Record config
    cfg = dict(
        comparisons=list(comparisons),
        main_type_col=main_type_col,
        compare_col=compare_col,
        lfc_threshold=lfc_threshold,
        pval_threshold=pval_threshold,
        gene_sets=list(gene_sets),
        organism=organism,
        outdir=outdir,
        min_cells=min_cells,
        top_n_terms=top_n_terms,
    )
    pd.Series(cfg).to_csv(os.path.join(outdir, "run_config.csv"))

    for group1, group2 in comparisons:
        comp_name = f"{group1}_vs_{group2}"
        safe_comp = safe_filename(comp_name)

        print(f"\n{'='*70}")
        print(f"Running: {comp_name}")
        print(f"{'='*70}")

        comp_dir = ensure_dir(os.path.join(outdir, safe_comp))
        comp_results = {}

        for celltype in all_celltypes:
            print(f"Processing celltype: {celltype}")
            safe_ct = safe_filename(celltype)

            # subset by celltype + groups
            adata_sub = adata[adata.obs[main_type_col] == celltype].copy()
            adata_sub = adata_sub[adata_sub.obs[compare_col].isin([group1, group2])].copy()

            if adata_sub.n_obs < min_cells:
                print(f"  Skipping: only {adata_sub.n_obs} cells (<{min_cells})")
                continue

            # DEG
            key = f"rank_genes_{safe_comp}"
            try:
                sc.tl.rank_genes_groups(
                    adata_sub,
                    groupby=compare_col,
                    groups=[group1],
                    reference=group2,
                    method='wilcoxon',
                    key_added=key
                )
            except Exception as e:
                print(f"  DEG failed: {e}")
                continue

            degs = sc.get.rank_genes_groups_df(adata_sub, group=group1, key=key)

            # Defensive: ensure expected columns exist
            required_cols = {'names', 'logfoldchanges', 'pvals_adj'}
            missing = required_cols - set(degs.columns)
            if missing:
                print(f"  DEG table missing columns {missing}; got {degs.columns.tolist()}")
                continue

            # Significant set
            sig = degs[(degs['pvals_adj'] < pval_threshold) &
                       (degs['logfoldchanges'].abs() > lfc_threshold)].copy()

            print(f"  Cells: {adata_sub.n_obs}")
            print(f"  Significant DEGs: {len(sig)}")

            # Save DEG tables
            deg_csv = os.path.join(comp_dir, f"{safe_ct}_DEG.csv")
            sig_csv = os.path.join(comp_dir, f"{safe_ct}_DEG_sig.csv")
            degs.to_csv(deg_csv, index=False)
            sig.to_csv(sig_csv, index=False)

            # Volcano
            volc_png = os.path.join(comp_dir, f"{safe_ct}_volcano.png")
            try:
                plot_volcano_publication_quality(
                    degs,
                    title=f"DEG: {celltype} ({group1} vs {group2})",
                    subtitle=f"Top genes in {celltype}",
                    lfc_threshold=lfc_threshold,
                    pval_threshold=pval_threshold,
                    savepath=volc_png
                )
            except Exception as e:
                print(f"  Volcano failed: {e}")

            # Enrichr ORA: up/down lists
            up = sig.loc[sig['logfoldchanges'] > lfc_threshold, 'names'].dropna().astype(str).tolist()
            down = sig.loc[sig['logfoldchanges'] < -lfc_threshold, 'names'].dropna().astype(str).tolist()

            enrichr_tables = {}

            # Up
            if len(up) > 5:
                try:
                    enr_up = gp.enrichr(
                        gene_list=up,
                        gene_sets=list(gene_sets),
                        organism=organism,
                        outdir=None
                    )
                    if enr_up is not None and enr_up.results is not None and not enr_up.results.empty:
                        up_res = enr_up.results.copy()
                        up_csv = os.path.join(comp_dir, f"{safe_ct}_Enrichr_up.csv")
                        up_res.to_csv(up_csv, index=False)

                        up_fig = os.path.join(comp_dir, f"{safe_ct}_Enrichr_up_dotplot.png")
                        enrichr_dotplot(
                            up_res,
                            title=f"Enrichr Up: {celltype} ({group1} vs {group2})",
                            top_n=top_n_terms,
                            savepath=up_fig
                        )

                        enrichr_tables['up'] = up_res
                        print(f"  Enrichr up: {len(up_res)} terms")
                    else:
                        print("  Enrichr up: empty results")
                except Exception as e:
                    print(f"  Enrichr up failed: {e}")
            else:
                print(f"  Enrichr up skipped: only {len(up)} genes")

            # Down
            if len(down) > 5:
                try:
                    enr_down = gp.enrichr(
                        gene_list=down,
                        gene_sets=list(gene_sets),
                        organism=organism,
                        outdir=None
                    )
                    if enr_down is not None and enr_down.results is not None and not enr_down.results.empty:
                        down_res = enr_down.results.copy()
                        down_csv = os.path.join(comp_dir, f"{safe_ct}_Enrichr_down.csv")
                        down_res.to_csv(down_csv, index=False)

                        down_fig = os.path.join(comp_dir, f"{safe_ct}_Enrichr_down_dotplot.png")
                        enrichr_dotplot(
                            down_res,
                            title=f"Enrichr Down: {celltype} ({group1} vs {group2})",
                            top_n=top_n_terms,
                            savepath=down_fig
                        )

                        enrichr_tables['down'] = down_res
                        print(f"  Enrichr down: {len(down_res)} terms")
                    else:
                        print("  Enrichr down: empty results")
                except Exception as e:
                    print(f"  Enrichr down failed: {e}")
            else:
                print(f"  Enrichr down skipped: only {len(down)} genes")

            comp_results[celltype] = {
                'degs': degs,
                'sig_degs': sig,
                'up_genes': up,
                'down_genes': down,
                'enrichr': enrichr_tables,
                'paths': {
                    'degs_csv': deg_csv,
                    'sig_csv': sig_csv,
                    'volcano_png': volc_png,
                }
            }

        all_results[comp_name] = comp_results

    print("\nAll comparisons complete!")
    return all_results

# ------------------------------------------------------------
# 3) Run (your parameters)
# ------------------------------------------------------------

comparisons = [
    ('R301', 'PBS'),
]

main_type_col = 'celltype'
compare_col = 'group'
lfc_threshold = 0.5
pval_threshold = 0.05
gene_sets = ['GO_Biological_Process_2023']
organism = 'mouse'
outdir = './DEG_Enrichr_all_comparisons'

all_results = run_deg_volcano_enrichr_all(
    adata=adata,
    comparisons=comparisons,
    main_type_col=main_type_col,
    compare_col=compare_col,
    lfc_threshold=lfc_threshold,
    pval_threshold=pval_threshold,
    gene_sets=gene_sets,
    organism=organism,
    outdir=outdir,
    min_cells=20,
    top_n_terms=15,
)

# Optional: quick summary
summary_rows = []
for comp, d in all_results.items():
    for ct, res in d.items():
        summary_rows.append({
            'comparison': comp,
            'celltype': ct,
            'n_deg_total': len(res['degs']),
            'n_deg_sig': len(res['sig_degs']),
            'n_up': len(res['up_genes']),
            'n_down': len(res['down_genes']),
            'has_enrichr_up': 'up' in res['enrichr'],
            'has_enrichr_down': 'down' in res['enrichr'],
        })
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(os.path.join(outdir, "summary.csv"), index=False)
summary_df.head()

### 6.6 Summarize

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ----------------------------
# Config
# ----------------------------
OUTDIR = "./DEG_Enrichr_all_comparisons"  # 改成你的实际输出目录
PVAL_THRESHOLD = 0.05
TOP_TERMS_PER_CONDITION = 15     # 每个 celltype×comparison×direction 取前N个term用于热图
TOP_TERMS_FREQUENCY = 25         # 频次柱状图展示前N个term
FIG_DPI = 300

# ----------------------------
# Helpers
# ----------------------------
def _read_csv_safe(path):
    try:
        return pd.read_csv(path)
    except Exception:
        return None

def _infer_celltype_from_filename(fname):
    # expects: <celltype>_DEG_sig.csv or <celltype>_Enrichr_up.csv
    base = os.path.basename(fname)
    for suffix in ["_DEG_sig.csv", "_DEG.csv", "_Enrichr_up.csv", "_Enrichr_down.csv"]:
        if base.endswith(suffix):
            return base[: -len(suffix)]
    return os.path.splitext(base)[0]

def ensure_dir(p):
    os.makedirs(p, exist_ok=True)
    return p

# ----------------------------
# 1) Summarize DEG counts
# ----------------------------
def summarize_deg_counts(outdir):
    rows = []

    comp_dirs = [d for d in glob.glob(os.path.join(outdir, "*")) if os.path.isdir(d)]
    for comp_dir in sorted(comp_dirs):
        comp_name = os.path.basename(comp_dir)

        sig_files = glob.glob(os.path.join(comp_dir, "*_DEG_sig.csv"))
        # fallback: if sig not saved, we can compute from DEG.csv (less ideal)
        if not sig_files:
            sig_files = glob.glob(os.path.join(comp_dir, "*_DEG.csv"))

        for f in sorted(sig_files):
            celltype = _infer_celltype_from_filename(f)
            df = _read_csv_safe(f)
            if df is None or df.empty:
                continue

            # Determine whether it's sig table or full DEG table
            is_sig = f.endswith("_DEG_sig.csv")

            # Need at least logfoldchanges, pvals_adj if full DEG
            if not is_sig:
                required = {"logfoldchanges", "pvals_adj"}
                if not required.issubset(df.columns):
                    continue
                df_sig = df[(df["pvals_adj"] < PVAL_THRESHOLD)].copy()
            else:
                df_sig = df.copy()

            if "logfoldchanges" not in df_sig.columns:
                continue

            n_sig = len(df_sig)
            n_up = int((df_sig["logfoldchanges"] > 0).sum())
            n_down = int((df_sig["logfoldchanges"] < 0).sum())

            rows.append({
                "comparison": comp_name,
                "celltype": celltype,
                "n_sig": n_sig,
                "n_up": n_up,
                "n_down": n_down,
                "sig_source": "DEG_sig" if is_sig else "DEG_filtered_by_padj"
            })

    res = pd.DataFrame(rows)
    return res

def plot_deg_counts_heatmaps(deg_counts_df, outdir_fig):
    ensure_dir(outdir_fig)

    if deg_counts_df is None or deg_counts_df.empty:
        print("No DEG summary data to plot.")
        return

    for metric, title in [
        ("n_sig", "Significant DEG counts (Up+Down)"),
        ("n_up", "Upregulated DEG counts"),
        ("n_down", "Downregulated DEG counts"),
    ]:
        mat = deg_counts_df.pivot_table(
            index="celltype", columns="comparison", values=metric, aggfunc="max", fill_value=0
        )

        plt.figure(figsize=(max(8, 0.6 * mat.shape[1] + 3), max(6, 0.35 * mat.shape[0] + 2)))
        ax = sns.heatmap(mat, cmap="mako", linewidths=0.3, linecolor="white")
        ax.set_title(title, fontsize=14, weight="bold")
        ax.set_xlabel("")
        ax.set_ylabel("")
        plt.tight_layout()
        path = os.path.join(outdir_fig, f"DEG_counts_heatmap_{metric}.png")
        plt.savefig(path, dpi=FIG_DPI)
        plt.close()

    # Also save a long-format barplot-friendly table
    deg_counts_df.to_csv(os.path.join(outdir_fig, "summary_deg_counts.csv"), index=False)
    print(f"Saved DEG summaries to: {outdir_fig}")

# ----------------------------
# 2) Summarize Enrichr results (Up/Down)
# ----------------------------
def summarize_enrichr(outdir):
    """
    Returns long table:
      comparison, celltype, direction(up/down), term, q, combined_score, overlap, genes...
    """
    rows = []
    comp_dirs = [d for d in glob.glob(os.path.join(outdir, "*")) if os.path.isdir(d)]

    for comp_dir in sorted(comp_dirs):
        comp_name = os.path.basename(comp_dir)

        for direction in ["up", "down"]:
            files = glob.glob(os.path.join(comp_dir, f"*_Enrichr_{direction}.csv"))
            for f in sorted(files):
                celltype = _infer_celltype_from_filename(f)
                df = _read_csv_safe(f)
                if df is None or df.empty:
                    continue

                # standard enrichr columns
                # Must have term + adjusted pvalue; be defensive
                term_col = "Term" if "Term" in df.columns else df.columns[0]

                q_col = "Adjusted P-value" if "Adjusted P-value" in df.columns else None
                if q_col is None:
                    for c in df.columns:
                        lc = c.lower()
                        if ("adjust" in lc or "adj" in lc) and ("p" in lc):
                            q_col = c
                            break
                if q_col is None:
                    continue

                score_col = "Combined Score" if "Combined Score" in df.columns else None
                if score_col is None:
                    for c in df.columns:
                        if "score" in c.lower():
                            score_col = c
                            break

                overlap_col = "Overlap" if "Overlap" in df.columns else None
                genes_col = "Genes" if "Genes" in df.columns else None

                tmp = df.copy()
                tmp = tmp.rename(columns={term_col: "term", q_col: "q"})
                tmp["comparison"] = comp_name
                tmp["celltype"] = celltype
                tmp["direction"] = direction

                if score_col and score_col in tmp.columns:
                    tmp = tmp.rename(columns={score_col: "combined_score"})
                else:
                    tmp["combined_score"] = np.nan

                if overlap_col and overlap_col in tmp.columns:
                    tmp = tmp.rename(columns={overlap_col: "overlap"})
                else:
                    tmp["overlap"] = np.nan

                if genes_col and genes_col in tmp.columns:
                    tmp = tmp.rename(columns={genes_col: "genes"})
                else:
                    tmp["genes"] = np.nan

                keep = ["comparison", "celltype", "direction", "term", "q", "combined_score", "overlap", "genes"]
                tmp = tmp[keep].copy()

                # ensure numeric q
                tmp["q"] = pd.to_numeric(tmp["q"], errors="coerce")
                tmp["combined_score"] = pd.to_numeric(tmp["combined_score"], errors="coerce")

                rows.append(tmp)

    if not rows:
        return pd.DataFrame(columns=["comparison","celltype","direction","term","q","combined_score","overlap","genes"])

    res = pd.concat(rows, ignore_index=True)
    return res

def plot_enrichr_frequency_and_heatmap(enr_long_df, outdir_fig):
    ensure_dir(outdir_fig)

    if enr_long_df is None or enr_long_df.empty:
        print("No Enrichr summary data to plot.")
        return

    # Save long table
    enr_long_df.to_csv(os.path.join(outdir_fig, "summary_enrichr_long.csv"), index=False)

    # Add -log10q
    df = enr_long_df.copy()
    df = df.dropna(subset=["q", "term"])
    df["-log10q"] = -np.log10(df["q"].astype(float) + 1e-300)

    for direction in ["up", "down"]:
        d = df[df["direction"] == direction].copy()
        if d.empty:
            continue

        # ---- Frequency barplot: how many (celltype×comparison) a term appears with q<PVAL_THRESHOLD
        d_sig = d[d["q"] < PVAL_THRESHOLD].copy()
        if d_sig.empty:
            continue

        freq = (d_sig.groupby("term")
                    .size()
                    .sort_values(ascending=False)
                    .head(TOP_TERMS_FREQUENCY))

        plt.figure(figsize=(10, max(5, 0.35 * len(freq) + 1.5)))
        ax = sns.barplot(x=freq.values, y=freq.index, color="#4C78A8")
        ax.set_title(f"Enrichr term frequency (q<{PVAL_THRESHOLD}) [{direction}]", fontsize=14, weight="bold")
        ax.set_xlabel("Count of (celltype × comparison) where term is significant")
        ax.set_ylabel("")
        plt.tight_layout()
        plt.savefig(os.path.join(outdir_fig, f"Enrichr_frequency_{direction}.png"), dpi=FIG_DPI)
        plt.close()

        # ---- Heatmap: term × condition (celltype|comparison) values = max(-log10q) among duplicates
        # pick top terms overall (by frequency) to keep heatmap readable
        top_terms = freq.index.tolist()

        d2 = d_sig[d_sig["term"].isin(top_terms)].copy()
        d2["condition"] = d2["celltype"].astype(str) + " | " + d2["comparison"].astype(str)

        mat = d2.pivot_table(
            index="term", columns="condition", values="-log10q", aggfunc="max", fill_value=0
        )

        # sort columns by comparison then celltype for readability
        # (optional) keep as-is; pivot_table column order is lexical
        plt.figure(figsize=(max(10, 0.25 * mat.shape[1] + 4), max(6, 0.35 * mat.shape[0] + 2)))
        ax = sns.heatmap(mat, cmap="rocket_r", linewidths=0.2, linecolor="white")
        ax.set_title(f"Enrichr heatmap: -log10(q) [{direction}] (top terms by frequency)", fontsize=14, weight="bold")
        ax.set_xlabel("")
        ax.set_ylabel("")
        plt.tight_layout()
        plt.savefig(os.path.join(outdir_fig, f"Enrichr_heatmap_{direction}.png"), dpi=FIG_DPI)
        plt.close()

    print(f"Saved Enrichr summaries to: {outdir_fig}")

# ----------------------------
# 3) (Optional) Term heatmap per comparison (more focused)
# ----------------------------
def plot_enrichr_heatmap_per_comparison(enr_long_df, outdir_fig, direction="up"):
    """
    For each comparison separately:
      term × celltype heatmap, using top terms within that comparison.
    """
    ensure_dir(outdir_fig)
    if enr_long_df is None or enr_long_df.empty:
        return

    df = enr_long_df.dropna(subset=["q", "term"]).copy()
    df["-log10q"] = -np.log10(df["q"].astype(float) + 1e-300)
    df = df[(df["direction"] == direction) & (df["q"] < PVAL_THRESHOLD)].copy()
    if df.empty:
        return

    for comp, dcomp in df.groupby("comparison"):
        # pick top terms inside this comparison (by median -log10q)
        term_rank = (dcomp.groupby("term")["-log10q"].median().sort_values(ascending=False))
        top_terms = term_rank.head(TOP_TERMS_PER_CONDITION).index.tolist()

        d2 = dcomp[dcomp["term"].isin(top_terms)]
        mat = d2.pivot_table(index="term", columns="celltype", values="-log10q", aggfunc="max", fill_value=0)

        plt.figure(figsize=(max(10, 0.35 * mat.shape[1] + 3), max(5, 0.4 * mat.shape[0] + 2)))
        ax = sns.heatmap(mat, cmap="rocket_r", linewidths=0.2, linecolor="white")
        ax.set_title(f"{comp}: Enrichr -log10(q) heatmap [{direction}]", fontsize=13, weight="bold")
        ax.set_xlabel("")
        ax.set_ylabel("")
        plt.tight_layout()
        plt.savefig(os.path.join(outdir_fig, f"Enrichr_heatmap_{direction}__{comp}.png"), dpi=FIG_DPI)
        plt.close()

# ----------------------------
# 4) Run everything
# ----------------------------
FIG_OUT = ensure_dir(os.path.join(OUTDIR, "_summary_figures"))

deg_counts = summarize_deg_counts(OUTDIR)
deg_counts.to_csv(os.path.join(FIG_OUT, "summary_deg_counts.csv"), index=False)
plot_deg_counts_heatmaps(deg_counts, FIG_OUT)

enr_long = summarize_enrichr(OUTDIR)
enr_long.to_csv(os.path.join(FIG_OUT, "summary_enrichr_long.csv"), index=False)
plot_enrichr_frequency_and_heatmap(enr_long, FIG_OUT)

# Optional: per-comparison term×celltype heatmaps (often更好读)
plot_enrichr_heatmap_per_comparison(enr_long, FIG_OUT, direction="up")
plot_enrichr_heatmap_per_comparison(enr_long, FIG_OUT, direction="down")

print("Done. Summary figures saved to:", FIG_OUT)

In [ ]:
# ==============================================================================
# Malignant-only pathway / module scoring
# ==============================================================================

import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

score_fig_dir = os.path.join(RESULTS_DIR, "figures", "malignant_module_scores")
os.makedirs(score_fig_dir, exist_ok=True)

# ------------------------------------------------------------------------------
# 1. Define malignant cell types
# ------------------------------------------------------------------------------

malignant_celltypes = [
    "Mesenchymal-like malignant cells",
    "Growth factor-high malignant cells",
    "Translation-high malignant cells",
    "Hypoxic/glycolytic malignant cells",
    "Cycling malignant cells",
    "IFN-stimulated malignant cells",
    "Proliferating malignant cells",
    "CDH13+ mesenchymal-like malignant cells"
]

malignant_celltypes = [
    ct for ct in malignant_celltypes
    if ct in adata.obs["celltype"].unique()
]

adata_tumor = adata[adata.obs["celltype"].isin(malignant_celltypes)].copy()

print(adata_tumor)
print(adata_tumor.obs["celltype"].value_counts())

In [ ]:
# ==============================================================================
# 2. Define module gene sets
# ==============================================================================

module_gene_sets = {
    # Proliferation / cell cycle
    "Proliferation": [
        "Mki67", "Top2a", "Prc1", "Ube2c", "Cenpf", "Cenpe",
        "Pbk", "Nusap1", "Kif11", "Kif20a", "Kif23", "Hmmr"
    ],

    "G2M_cell_cycle": [
        "Ccnb1", "Ccnb2", "Cdc20", "Cdc25c", "Plk1", "Aurka",
        "Aurkb", "Tpx2", "Cenpa", "Cenpe", "Cenpf", "Kif2c"
    ],

    "DNA_replication": [
        "Mcm2", "Mcm3", "Mcm4", "Mcm5", "Mcm6", "Mcm7",
        "Pcna", "Rrm1", "Rrm2", "Tyms", "Fen1", "Ung"
    ],

    # Translation / ribosome
    "Translation": [
        "Eif4a1", "Eif3a", "Eif3b", "Eif3e", "Eif4e",
        "Eef1a1", "Eef1b2", "Eef2", "Rplp0", "Rps3", "Rps6"
    ],

    "Ribosome": [
        "Rpl3", "Rpl4", "Rpl5", "Rpl7", "Rpl8", "Rpl10",
        "Rpl13", "Rpl13a", "Rplp0", "Rps3", "Rps6", "Rps8"
    ],

    # Hypoxia / metabolism
    "Hypoxia": [
        "P4ha1", "P4ha2", "Plod2", "Ankrd37", "Bnip3", "Bnip3l",
        "Ndrg1", "Pdk1", "Higd1a", "Vegfa", "Slc2a1", "Ldha"
    ],

    "Glycolysis": [
        "Hk1", "Hk2", "Gpi1", "Pfkl", "Pfkp", "Aldoa",
        "Gapdh", "Pgk1", "Pgam1", "Eno1", "Eno2", "Pkm", "Ldha"
    ],

    "OXPHOS": [
        "Ndufa1", "Ndufa2", "Ndufa4", "Ndufb5", "Ndufs1",
        "Sdha", "Uqcrc1", "Cox4i1", "Cox5a", "Atp5f1a", "Atp5f1b"
    ],

    # IFN / inflammation
    "IFN_response": [
        "Ifit1", "Ifit2", "Ifit3", "Ifit3b", "Isg15", "Usp18",
        "Oasl2", "Oas1a", "Oas2", "Rsad2", "Ifih1", "Mx1", "Stat1"
    ],

    "Inflammatory_response": [
        "Il1b", "Il6", "Tnf", "Cxcl1", "Cxcl2", "Cxcl3",
        "Ccl2", "Ccl7", "Ptgs2", "Nfkbia", "Socs3"
    ],

    # EMT / mesenchymal-like
    "EMT_mesenchymal": [
        "Vim", "Fn1", "Col1a1", "Col1a2", "Col3a1", "Col5a2",
        "Prrx1", "Zeb1", "Zeb2", "Snai1", "Snai2", "Twist1",
        "Cdh13", "Sema3a", "Slit2"
    ],

    "ECM_remodeling": [
        "Col1a1", "Col1a2", "Col3a1", "Col5a2", "Col6a2",
        "Col6a3", "Mmp2", "Mmp9", "Mmp14", "Lox", "Loxl2", "Fbn1"
    ],

    # Growth factor / receptor signaling
    "Growth_factor_signaling": [
        "Ereg", "Hbegf", "Areg", "Btc", "Epgn", "Grem1",
        "Timp1", "Egfr", "Erbb2", "Met", "Igf1r", "Itga6"
    ],

    "MAPK_EGFR_related": [
        "Ereg", "Hbegf", "Areg", "Egfr", "Dusp1", "Dusp4",
        "Dusp6", "Fos", "Jun", "Egr1", "Spry2", "Spry4"
    ],

    # Antigen presentation / immune visibility
    "MHC_I_antigen_presentation": [
        "B2m", "H2-K1", "H2-D1", "Tap1", "Tap2", "Tapbp",
        "Psmb8", "Psmb9", "Psmb10", "Nlrc5"
    ],

    "MHC_II_antigen_presentation": [
        "H2-Aa", "H2-Ab1", "H2-Eb1", "H2-Ea", "Cd74",
        "Ciita", "H2-DMa", "H2-DMb1", "H2-DMb2"
    ],

    # Stress / death
    "Apoptosis": [
        "Bax", "Bak1", "Bbc3", "Pmaip1", "Casp3", "Casp7",
        "Casp8", "Casp9", "Fas", "Fasl", "Tnfrsf10b"
    ],

    "p53_pathway": [
        "Trp53", "Cdkn1a", "Mdm2", "Bax", "Bbc3", "Pmaip1",
        "Gadd45a", "Sesn1", "Sesn2", "Rrm2b"
    ],

    "DNA_damage_response": [
        "Atm", "Atr", "Chek1", "Chek2", "Brca1", "Brca2",
        "Rad51", "H2afx", "Gadd45a", "Ddb2", "Xrcc5"
    ],

    # Tumor-neutrophil axis
    "Neutrophil_recruitment": [
        "Cxcl1", "Cxcl2", "Cxcl3", "Cxcl5", "Csf3",
        "Il1b", "Ccl2", "Ccl7", "Lcn2", "S100a8", "S100a9"
    ]
}

In [ ]:
# ==============================================================================
# 3. Score modules in malignant cells
# ==============================================================================

def filter_genes(gene_list, adata_obj):
    """Keep genes present in adata.var_names."""
    return [g for g in gene_list if g in adata_obj.var_names]

score_names = []

for module_name, genes in module_gene_sets.items():
    genes_use = filter_genes(genes, adata_tumor)
    
    if len(genes_use) < 3:
        print(f"Skip {module_name}: only {len(genes_use)} genes found.")
        continue
    
    score_name = module_name + "_score"
    
    sc.tl.score_genes(
        adata_tumor,
        gene_list=genes_use,
        score_name=score_name,
        ctrl_size=min(50, len(genes_use) * 2),
        random_state=0,
        use_raw=True
    )
    
    score_names.append(score_name)
    print(f"{module_name}: {len(genes_use)} genes used.")

print("\nScores added:")
print(score_names)

In [ ]:
# ==============================================================================
# 4. Summarize module scores by group and malignant state
# ==============================================================================

# Mean score by treatment group
score_by_group = (
    adata_tumor.obs
    .groupby("group")[score_names]
    .mean()
)

# Mean score by malignant cell state
score_by_celltype = (
    adata_tumor.obs
    .groupby("celltype")[score_names]
    .mean()
)

# Mean score by group and malignant state
score_by_group_celltype = (
    adata_tumor.obs
    .groupby(["group", "celltype"])[score_names]
    .mean()
)

print("Score by group:")
display(score_by_group.round(3))

print("Score by malignant cell state:")
display(score_by_celltype.round(3))

In [ ]:
score_by_group.to_csv(os.path.join(score_fig_dir, "ModuleScores_by_group.csv"))
score_by_celltype.to_csv(os.path.join(score_fig_dir, "ModuleScores_by_malignant_celltype.csv"))
score_by_group_celltype.to_csv(os.path.join(score_fig_dir, "ModuleScores_by_group_and_malignant_celltype.csv"))

In [ ]:
# ==============================================================================
# 5. Violin plots: PBS vs R301 in malignant cells
# ==============================================================================

selected_scores = [
    "Proliferation_score",
    "G2M_cell_cycle_score",
    "Translation_score",
    "Hypoxia_score",
    "Glycolysis_score",
    "IFN_response_score",
    "EMT_mesenchymal_score",
    "Growth_factor_signaling_score",
    "MHC_I_antigen_presentation_score",
    "Neutrophil_recruitment_score"
]

selected_scores = [s for s in selected_scores if s in adata_tumor.obs.columns]

sc.pl.violin(
    adata_tumor,
    keys=selected_scores,
    groupby="group",
    rotation=45,
    stripplot=False,
    multi_panel=True,
    show=False
)

plt.savefig(
    os.path.join(score_fig_dir, "Violin_ModuleScores_by_group_malignant_only.pdf"),
    dpi=300,
    bbox_inches="tight"
)
plt.show()
plt.close()

In [ ]:
# ==============================================================================
# 6. Dotplot: module scores by malignant cell state
# ==============================================================================

# Scanpy dotplot一般用于基因表达。
# 对obs里的score，更方便用matrixplot-like heatmap，见下一段。
# 这里先画 UMAP feature plot。

for score in selected_scores:
    sc.pl.umap(
        adata_tumor,
        color=score,
        cmap="viridis",
        show=False
    )
    
    plt.savefig(
        os.path.join(score_fig_dir, f"UMAP_{score}.pdf"),
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()
    plt.close()

In [ ]:
# ==============================================================================
# 8. Heatmap: ALL computed module scores across treatment groups
# ==============================================================================

# Use ALL module score columns (not just selected_scores subset)
all_module_score_cols = [c for c in adata_tumor.obs.columns if c.endswith('_score')]
# Exclude generic QC scores, keep only pathway modules
exclude_scores = ['S_score', 'G2M_score', 'doubletdetection_score', 'heuristic_confidence_score', 'cnv_score']
all_module_score_cols = [c for c in all_module_score_cols if c not in exclude_scores]
all_module_score_cols = sorted(all_module_score_cols)

print(f"Plotting {len(all_module_score_cols)} module scores: {all_module_score_cols}")

group_heat_df = (
    adata_tumor.obs
    .groupby("group")[all_module_score_cols]
    .mean()
)

# Reorder groups if treatment_order exists
if "treatment_order" in globals():
    group_heat_df = group_heat_df.reindex([g for g in treatment_order if g in group_heat_df.index])

group_heat_df_z = (group_heat_df - group_heat_df.mean(axis=0)) / group_heat_df.std(axis=0)
group_heat_df_z = group_heat_df_z.fillna(0)

# Larger figure for all modules
fig, ax = plt.subplots(figsize=(14, 3))

im = ax.imshow(group_heat_df_z.values, aspect="auto", cmap="RdBu_r", vmin=-2, vmax=2)

ax.set_xticks(np.arange(len(group_heat_df_z.columns)))
ax.set_xticklabels(
    [c.replace("_score", "").replace("_", " ") for c in group_heat_df_z.columns],
    rotation=45,
    ha="right",
    fontsize=8
)

ax.set_yticks(np.arange(len(group_heat_df_z.index)))
ax.set_yticklabels(group_heat_df_z.index, fontsize=10)

ax.set_title("All Pathway Module Changes in Malignant Cells (R301 vs PBS)", fontsize=13, fontweight="bold")

cbar = plt.colorbar(im, ax=ax, fraction=0.02, pad=0.02)
cbar.set_label("Z-scored mean module score")

plt.tight_layout()

plt.savefig(
    os.path.join(score_fig_dir, "Heatmap_ALL_ModuleScores_by_group_malignant.pdf"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

# Also save the raw mean values for reference
group_heat_df.to_csv(os.path.join(score_fig_dir, "ModuleScores_by_group_raw_values.csv"))
print(f"Saved comprehensive heatmap to: {score_fig_dir}")



## 7. Advanced Treatment-Response Analyses

针对三抗（R301）治疗进行免疫微环境重塑、跨细胞类型联合分析和功能模块全 compartments 比较。

### 7.1 Immune Infiltration & TME Remodeling (Sample-Level)

计算每个样本的免疫浸润分数、基质分数及其亚组分（T细胞、NK、髓系、细胞毒性），
在样本水平比较 PBS vs R301。

In [ ]:
# ==============================================================================
# 7.1 Immune Infiltration & TME Remodeling (Sample-Level)
# ==============================================================================

import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

tme_dir = os.path.join(RESULTS_DIR, 'tme_remodeling')
os.makedirs(tme_dir, exist_ok=True)

# --------------------------------------------------------------------------
# Define gene signatures (mouse)
# --------------------------------------------------------------------------
signatures = {
    'Immune_Score': [
        'Ptprc', 'Cd3d', 'Cd3e', 'Cd4', 'Cd8a', 'Cd8b1', 'Cd19', 'Cd79a', 'Cd79b',
        'Ncr1', 'Nkg7', 'Cd14', 'Cd68', 'Itgam', 'Itgax', 'Fcgr3',
        'Csf1r', 'Cxcr3', 'Ccr7', 'Il2rb', 'Gzmb', 'Prf1'
    ],
    'Stromal_Score': [
        'Col1a1', 'Col1a2', 'Col3a1', 'Col5a2', 'Col6a2', 'Fbn1', 'Dcn', 'Lum',
        'Mmp2', 'Timp1', 'Acta2', 'Tagln', 'Pecam1', 'Cdh5', 'Eng', 'Vwf',
        'Rgs5', 'Pdgfrb', 'Cspg4'
    ],
    'T_Cell_Score': [
        'Cd3d', 'Cd3e', 'Cd4', 'Cd8a', 'Cd8b1', 'Cd2', 'Cd28', 'Trac', 'Trbc1', 'Trbc2'
    ],
    'NK_Cell_Score': [
        'Ncr1', 'Nkg7', 'Klrb1c', 'Klrd1', 'Prf1', 'Gzmb', 'Ctsw', 'Txk'
    ],
    'Myeloid_Score': [
        'Cd14', 'Cd68', 'Itgam', 'Csf1r', 'Fcgr1', 'Fcgr3', 'Adgre1', 'Mertk', 'Trem2', 'Aif1'
    ],
    'Cytotoxic_Score': [
        'Gzma', 'Gzmb', 'Gzmk', 'Prf1', 'Ctsw', 'Nkg7', 'Ifng'
    ],
}

# Score each cell using adata.raw
for sig_name, genes in signatures.items():
    genes_avail = [g for g in genes if g in adata.raw.var_names]
    if len(genes_avail) >= 3:
        sc.tl.score_genes(
            adata, gene_list=genes_avail, score_name=sig_name,
            use_raw=True, ctrl_size=min(50, len(genes_avail) * 2), random_state=42
        )
        print(f'Scored {sig_name}: {len(genes_avail)}/{len(genes)} genes')

score_cols = [s for s in signatures.keys() if s in adata.obs.columns]
print(f'\nTotal score columns added: {len(score_cols)}')

In [ ]:
# --------------------------------------------------------------------------
# Aggregate to sample level & visualize
# --------------------------------------------------------------------------

# Per-sample mean scores
sample_scores = adata.obs.groupby('sampleID')[score_cols].mean()
sample_to_group = adata.obs[['sampleID', 'group']].drop_duplicates().set_index('sampleID')
sample_scores = sample_scores.join(sample_to_group)
sample_scores = sample_scores.sort_values('group')

existing_groups = [g for g in ['PBS', 'R301'] if g in sample_scores['group'].unique()]

# ---- A) Paired boxplots with MWU p-values ----
n_cols = 3
n_rows = int(np.ceil(len(score_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
axes = axes.flatten()

mw_results = []
for i, sc_name in enumerate(score_cols):
    ax = axes[i]
    # Boxplot
    sns.boxplot(data=sample_scores, x='group', y=sc_name, order=existing_groups,
                palette=color, width=0.5, ax=ax)
    sns.stripplot(data=sample_scores, x='group', y=sc_name, order=existing_groups,
                   color='black', size=7, jitter=0.12, alpha=0.85, ax=ax)
    
    # MWU test
    a = sample_scores.loc[sample_scores['group'] == 'R301', sc_name].values
    b = sample_scores.loc[sample_scores['group'] == 'PBS', sc_name].values
    if len(a) >= 2 and len(b) >= 2:
        u, p = mannwhitneyu(a, b, alternative='two-sided')
        mw_results.append({'score': sc_name, 'u_stat': u, 'p_value': p})
    else:
        mw_results.append({'score': sc_name, 'u_stat': np.nan, 'p_value': np.nan})
    
    ax.set_title(sc_name.replace('_', ' '), fontsize=11)
    ax.set_xlabel('')
    ax.set_ylabel('Mean Score per Sample')

# Hide unused axes
for j in range(len(score_cols), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(tme_dir, 'TME_Scores_boxplot.pdf'), dpi=300, bbox_inches='tight')
plt.show()
plt.close()

# FDR correction
mw_df = pd.DataFrame(mw_results)
mw_df['p_fdr'] = multipletests(mw_df['p_value'].fillna(1.0).values, method='fdr_bh')[1]
mw_df.to_csv(os.path.join(tme_dir, 'TME_Scores_MWU.csv'), index=False)
print('\nTME Score Comparison (R301 vs PBS, MWU + FDR):')
print(mw_df.round(4).to_string(index=False))

In [ ]:
# ---- B) Immune-Stromal balance barplot per sample ----
fig, ax = plt.subplots(figsize=(9, 5))

plot_df = sample_scores[['Immune_Score', 'Stromal_Score']].copy()
plot_df = plot_df.join(sample_to_group)
plot_df = plot_df.sort_values('group')

x = range(len(plot_df))
width = 0.35
labels = [f'{r.name}\n({r["group"]})' for _, r in plot_df.iterrows()]

ax.bar([i - width/2 for i in x], plot_df['Immune_Score'], width,
       label='Immune Score', color='#3498db', alpha=0.85)
ax.bar([i + width/2 for i in x], plot_df['Stromal_Score'], width,
       label='Stromal Score', color='#e74c3c', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Mean Score per Sample')
ax.set_title('Immune vs Stromal Score by Sample', fontsize=13, fontweight='bold')
ax.legend(loc='upper right', frameon=False)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(tme_dir, 'Immune_Stromal_balance_barplot.pdf'), dpi=300, bbox_inches='tight')
plt.show()
plt.close()

# ---- C) Heatmap: Z-scored scores across samples ----
z_df = (sample_scores[score_cols] - sample_scores[score_cols].mean()) / sample_scores[score_cols].std()
z_df = z_df.fillna(0)

fig, ax = plt.subplots(figsize=(10, 3.5))
im = ax.imshow(z_df.T.values, aspect='auto', cmap='RdBu_r', vmin=-2, vmax=2)
ax.set_xticks(range(len(z_df)))
ax.set_xticklabels([f'{r.name} ({r["group"]})' for _, r in plot_df.iterrows()],
                   rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(score_cols)))
ax.set_yticklabels([s.replace('_Score', '') for s in score_cols], fontsize=9)
ax.set_title('Z-scored TME Signatures by Sample', fontsize=13, fontweight='bold')
cbar = plt.colorbar(im, ax=ax, fraction=0.02, pad=0.02)
cbar.set_label('Z-score')
plt.tight_layout()
plt.savefig(os.path.join(tme_dir, 'TME_Scores_heatmap.pdf'), dpi=300, bbox_inches='tight')
plt.show()
plt.close()

print(f'TME remodeling figures saved to: {tme_dir}')

### 7.2 Cross-Celltype Shared DEG Analysis

从每种细胞类型的 DEG 结果中，找出在多种细胞类型中共同上调或下调的基因。
这些跨细胞类型的共享 DEG 可能代表三抗治疗的全局性转录重编程效应。

In [ ]:
# ==============================================================================
# 7.2 Cross-Celltype Shared DEG Analysis
# ==============================================================================

import glob
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gseapy as gp

shared_dir = os.path.join(RESULTS_DIR, 'shared_degs')
os.makedirs(shared_dir, exist_ok=True)

# --------------------------------------------------------------------------
# 1) Load all per-celltype significant DEGs
# --------------------------------------------------------------------------
DEG_DIR = os.path.join(BASE_DIR, 'DEG_Enrichr_all_comparisons/R301_vs_PBS')
sig_files = sorted(glob.glob(os.path.join(DEG_DIR, '*_DEG_sig.csv')))
print(f'Found {len(sig_files)} significant DEG files')

LFC_THRESHOLD = 0.5
PVAL_THRESHOLD = 0.05

# Parse celltype from filename
def parse_celltype(filename):
    base = os.path.basename(filename)
    # Remove _DEG_sig.csv suffix
    ct = base.replace('_DEG_sig.csv', '').replace('_', ' ').replace('  ', ' ')
    return ct

# Collect all DEGs
deg_by_celltype = {}
for f in sig_files:
    ct = parse_celltype(f)
    df = pd.read_csv(f)
    # Standardize column names
    if 'logfoldchanges' not in df.columns:
        rename_map = {}
        for c in df.columns:
            if 'log' in c.lower() and 'fc' in c.lower():
                rename_map[c] = 'logfoldchanges'
            elif 'padj' in c.lower() or 'pval' in c.lower() or 'adj' in c.lower():
                rename_map[c] = 'pvals_adj'
        df = df.rename(columns=rename_map)
    
    if 'logfoldchanges' not in df.columns or 'pvals_adj' not in df.columns:
        print(f'  Skipping {ct}: missing required columns')
        continue
    
    df['logfoldchanges'] = pd.to_numeric(df['logfoldchanges'], errors='coerce')
    df['pvals_adj'] = pd.to_numeric(df['pvals_adj'], errors='coerce')
    
    # Significant genes
    sig = df[(df['pvals_adj'] < PVAL_THRESHOLD) & (df['logfoldchanges'].abs() > LFC_THRESHOLD)]
    if len(sig) == 0:
        continue
    
    deg_by_celltype[ct] = sig.copy()
    print(f'  {ct}: {len(sig)} sig DEGs ({(sig["logfoldchanges"] > 0).sum()} UP, {(sig["logfoldchanges"] < 0).sum()} DOWN)')

print(f'\nCell types with sig DEGs: {len(deg_by_celltype)}')

In [ ]:
# --------------------------------------------------------------------------
# 2) Count gene recurrence across cell types (UP and DOWN separately)
# --------------------------------------------------------------------------

# Build gene x celltype matrices
all_ct = list(deg_by_celltype.keys())
all_genes = set()
for ct, df in deg_by_celltype.items():
    gene_col = 'names' if 'names' in df.columns else df.columns[0]
    all_genes.update(df[gene_col].astype(str).tolist())
all_genes = sorted(all_genes)

gene_up_count = {}
gene_down_count = {}
gene_lfc = {}  # mean log2FC across cell types

for gene in all_genes:
    up_cts = []
    down_cts = []
    lfc_vals = []
    for ct, df in deg_by_celltype.items():
        gene_col = 'names' if 'names' in df.columns else df.columns[0]
        gene_df = df[df[gene_col].astype(str) == gene]
        if len(gene_df) > 0:
            lfc = gene_df['logfoldchanges'].values[0]
            lfc_vals.append(lfc)
            if lfc > 0:
                up_cts.append(ct)
            else:
                down_cts.append(ct)
    if up_cts:
        gene_up_count[gene] = (len(up_cts), up_cts)
    if down_cts:
        gene_down_count[gene] = (len(down_cts), down_cts)
    if lfc_vals:
        gene_lfc[gene] = np.mean(lfc_vals)

# Sort by frequency
up_sorted = sorted(gene_up_count.items(), key=lambda x: x[1][0], reverse=True)
down_sorted = sorted(gene_down_count.items(), key=lambda x: x[1][0], reverse=True)

print(f'Genes UP in >=1 cell type: {len(up_sorted)}')
print(f'Genes DOWN in >=1 cell type: {len(down_sorted)}')
print(f'\nTop UP genes shared across cell types:')
for g, (n, cts) in up_sorted[:20]:
    if n >= 3:
        print(f'  {g}: {n} cell types ({cts[:3]}...)')

print(f'\nTop DOWN genes shared across cell types:')
for g, (n, cts) in down_sorted[:20]:
    if n >= 3:
        print(f'  {g}: {n} cell types ({cts[:3]}...)')

In [ ]:
# --------------------------------------------------------------------------
# 3) Visualization: Frequency barplot of shared genes
# --------------------------------------------------------------------------

for direction, sorted_data, cmap_name, title_suffix in [
    ('UP', up_sorted, 'Reds', 'Up-regulated'),
    ('DOWN', down_sorted, 'Blues', 'Down-regulated'),
]:
    # Genes shared in >=2 cell types
    shared = [(g, n, cts) for g, (n, cts) in sorted_data if n >= 2]
    if len(shared) == 0:
        continue
    
    top_n = min(30, len(shared))
    top_shared = shared[:top_n]
    
    fig, ax = plt.subplots(figsize=(10, max(6, 0.35 * top_n + 1)))
    genes = [x[0] for x in top_shared]
    counts = [x[1] for x in top_shared]
    colors = plt.cm.get_cmap(cmap_name)(np.linspace(0.4, 0.9, top_n))
    
    bars = ax.barh(range(top_n), counts, color=colors, edgecolor='white', linewidth=0.5)
    ax.set_yticks(range(top_n))
    ax.set_yticklabels(genes, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel('Number of Cell Types', fontsize=12)
    ax.set_title(f'Top Shared {title_suffix} Genes Across Cell Types (R301 vs PBS)',
                 fontsize=13, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    
    # Add cell type names as annotation for top 5
    for i, (g, n, cts) in enumerate(top_shared[:5]):
        ct_str = ', '.join(cts[:5])
        if len(cts) > 5:
            ct_str += f' ...+{len(cts)-5}'
        ax.text(n + 0.1, i, ct_str, va='center', fontsize=7, color='#333333')
    
    plt.tight_layout()
    plt.savefig(os.path.join(shared_dir, f'Shared_DEG_frequency_{direction}.pdf'), dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    
    # Save table
    freq_df = pd.DataFrame([
        {'gene': g, 'n_celltypes': n, 'celltypes': ', '.join(cts), 'mean_log2FC': gene_lfc.get(g, np.nan)}
        for g, n, cts in shared
    ])
    freq_df.to_csv(os.path.join(shared_dir, f'Shared_DEG_{direction}_frequency.csv'), index=False)

In [ ]:
# --------------------------------------------------------------------------
# 4) Heatmap: log2FC of top shared genes across cell types
# --------------------------------------------------------------------------

for direction, sorted_data, cmap_name in [
    ('UP', up_sorted, 'RdBu_r'),
    ('DOWN', down_sorted, 'RdBu_r'),
]:
    shared = [(g, n, cts) for g, (n, cts) in sorted_data if n >= 3]
    if len(shared) < 3:
        continue
    
    # Top 25 shared genes
    top_genes = [x[0] for x in shared[:25]]
    
    # Build log2FC matrix: genes x cell types
    lfc_matrix = pd.DataFrame(index=top_genes, columns=all_ct, data=np.nan)
    for ct, df in deg_by_celltype.items():
        gene_col = 'names' if 'names' in df.columns else df.columns[0]
        for _, row in df.iterrows():
            g = str(row[gene_col])
            if g in top_genes:
                lfc_matrix.loc[g, ct] = row['logfoldchanges']
    
    # Drop rows and columns that are all NaN
    lfc_matrix = lfc_matrix.dropna(how='all', axis=0).dropna(how='all', axis=1)
    if lfc_matrix.empty:
        continue
    lfc_matrix = lfc_matrix.fillna(0)
    
    # Sort rows by sum of |LFC|, columns alphabetically
    lfc_matrix['abs_sum'] = lfc_matrix.abs().sum(axis=1)
    lfc_matrix = lfc_matrix.sort_values('abs_sum', ascending=False).drop(columns='abs_sum')
    
    fig, ax = plt.subplots(
        figsize=(max(10, 0.3 * lfc_matrix.shape[1] + 3),
                 max(5, 0.35 * lfc_matrix.shape[0] + 2))
    )
    sns.heatmap(lfc_matrix, cmap=cmap_name, center=0, linewidths=0.3, linecolor='white',
                cbar_kws={'label': 'log2 Fold Change (R301 vs PBS)'}, ax=ax)
    ax.set_title(f'{direction}-regulated DEGs Shared Across Cell Types',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('')
    plt.tight_layout()
    plt.savefig(os.path.join(shared_dir, f'Shared_DEG_heatmap_{direction}.pdf'), dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

In [ ]:
# --------------------------------------------------------------------------
# 5) GO enrichment of cross-celltype shared genes
# --------------------------------------------------------------------------

for direction, sorted_data, title_suffix in [
    ('UP', up_sorted, 'Up-regulated'),
    ('DOWN', down_sorted, 'Down-regulated'),
]:
    shared = [(g, n, cts) for g, (n, cts) in sorted_data if n >= 3]
    if len(shared) < 5:
        print(f'Too few shared {direction} genes for enrichment (need >=5, got {len(shared)})')
        continue
    
    shared_genes = [x[0] for x in shared]
    print(f'\nRunning GO enrichment for {len(shared_genes)} shared {direction} genes...')
    
    try:
        enr = gp.enrichr(
            gene_list=shared_genes,
            gene_sets=['GO_Biological_Process_2023'],
            organism='mouse',
            outdir=None,
        )
        if enr is not None and enr.results is not None and not enr.results.empty:
            res = enr.results.copy()
            res.to_csv(os.path.join(shared_dir, f'Shared_DEG_GO_enrichment_{direction}.csv'), index=False)
            
            # Dotplot of top 15 terms
            top = res.head(15).copy()
            top['Adjusted P-value'] = top['Adjusted P-value'].astype(float)
            top['-log10(q)'] = -np.log10(top['Adjusted P-value'] + 1e-300)
            top = top.sort_values('-log10(q)')
            
            fig, ax = plt.subplots(figsize=(10, max(4, 0.35 * len(top) + 1.5)))
            sca = ax.scatter(
                top['-log10(q)'], top['Term'],
                s=60 + 10 * top['Overlap'].str.split('/', expand=True)[0].astype(float),
                c=top['Combined Score'].astype(float) if 'Combined Score' in top.columns else top['-log10(q)'],
                cmap='viridis', alpha=0.9, edgecolor='none'
            )
            ax.set_xlabel('-log10(Adjusted P-value)', fontsize=12)
            ax.set_title(f'GO BP: Shared {title_suffix} Genes Across Cell Types',
                         fontsize=13, fontweight='bold')
            cbar = plt.colorbar(sca, ax=ax, pad=0.02)
            cbar.set_label('Combined Score')
            sns.despine(ax=ax, left=False, bottom=False)
            plt.tight_layout()
            plt.savefig(os.path.join(shared_dir, f'Shared_DEG_GO_enrichment_{direction}.pdf'),
                        dpi=300, bbox_inches='tight')
            plt.show()
            plt.close()
            print(f'  Top 5 GO terms:')
            for _, r in top.head(5).iterrows():
                print(f'    {r["Term"]}: q={r["Adjusted P-value"]:.2e}')
        else:
            print(f'  No enrichment results for {direction} genes')
    except Exception as e:
        print(f'  Enrichment failed for {direction}: {e}')

print(f'\nShared DEG analysis saved to: {shared_dir}')

### 7.3 Gene Signature Scoring at Sample Level

定义治疗相关的基因签名（免疫激活、T细胞耗竭、IFN响应、血管生成、ECM重塑、抗原呈递、炎症），
在细胞水平评分后聚合到样本水平，比较 PBS vs R301。

In [ ]:
# ==============================================================================
# 7.3 Gene Signature Scoring at Sample Level
# ==============================================================================

import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

sig_dir = os.path.join(RESULTS_DIR, 'sample_signatures')
os.makedirs(sig_dir, exist_ok=True)

# --------------------------------------------------------------------------
# Define treatment-relevant gene signatures
# --------------------------------------------------------------------------
treatment_signatures = {
    'Immune_Activation': [
        'Ifng', 'Gzmb', 'Prf1', 'Tbx21', 'Eomes', 'Cd69', 'Icos',
        'Il2ra', 'Tnf', 'Ccl3', 'Ccl4', 'Xcl1'
    ],
    'T_Cell_Exhaustion': [
        'Pdcd1', 'Lag3', 'Havcr2', 'Tigit', 'Ctla4', 'Tox', 'Entpd1', 'Eomes'
    ],
    'IFN_Response': [
        'Ifit1', 'Ifit3', 'Isg15', 'Stat1', 'Stat2', 'Irf7', 'Irf9',
        'Oasl2', 'Rsad2', 'Ifih1', 'Mx1', 'Usp18'
    ],
    'Angiogenesis': [
        'Vegfa', 'Vegfb', 'Kdr', 'Flt1', 'Angpt1', 'Angpt2', 'Tek',
        'Pdgfa', 'Pdgfb', 'Pdgfra', 'Pdgfrb', 'Nrp1'
    ],
    'ECM_Remodeling': [
        'Col1a1', 'Col3a1', 'Col5a2', 'Fbn1', 'Mmp2', 'Mmp9', 'Mmp14',
        'Lox', 'Loxl2', 'Fn1', 'Timp1', 'Postn'
    ],
    'Antigen_Presentation': [
        'B2m', 'H2-K1', 'H2-D1', 'H2-Aa', 'H2-Ab1', 'H2-Eb1',
        'Cd74', 'Tap1', 'Tap2', 'Tapbp', 'Ciita', 'Psmb8', 'Psmb9'
    ],
    'Inflammatory_Response': [
        'Il1b', 'Il6', 'Tnf', 'Cxcl1', 'Cxcl2', 'Cxcl3',
        'Ccl2', 'Ccl7', 'Ptgs2', 'Nfkbia', 'Socs3', 'Cxcl10'
    ],
}

# Score each cell
for sig_name, genes in treatment_signatures.items():
    genes_avail = [g for g in genes if g in adata.raw.var_names]
    if len(genes_avail) >= 3:
        sc.tl.score_genes(
            adata, gene_list=genes_avail, score_name=sig_name,
            use_raw=True, ctrl_size=min(50, len(genes_avail) * 2), random_state=42
        )
        print(f'Scored {sig_name}: {len(genes_avail)}/{len(genes)} genes')

sig_score_cols = [s for s in treatment_signatures.keys() if s in adata.obs.columns]
print(f'\n{len(sig_score_cols)} signature scores computed')

In [ ]:
# --------------------------------------------------------------------------
# Aggregate to sample level & visualize
# --------------------------------------------------------------------------

# Per-sample mean of signature scores
samp_sig = adata.obs.groupby('sampleID')[sig_score_cols].mean()
samp_sig = samp_sig.join(sample_to_group)
samp_sig = samp_sig.sort_values('group')

# ---- A) Heatmap: Z-scored signatures across samples ----
z_sig = (samp_sig[sig_score_cols] - samp_sig[sig_score_cols].mean()) / samp_sig[sig_score_cols].std()
z_sig = z_sig.fillna(0)

fig, ax = plt.subplots(figsize=(9, 4))
im = ax.imshow(z_sig.T.values, aspect='auto', cmap='RdBu_r', vmin=-2, vmax=2)
ax.set_xticks(range(len(z_sig)))
ax.set_xticklabels([f'{r.name} ({r["group"]})' for _, r in samp_sig.iterrows()],
                   rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(sig_score_cols)))
ax.set_yticklabels([s.replace('_', ' ') for s in sig_score_cols], fontsize=10)
ax.set_title('Treatment-Associated Gene Signatures by Sample', fontsize=13, fontweight='bold')
cbar = plt.colorbar(im, ax=ax, fraction=0.02, pad=0.02)
cbar.set_label('Z-score')
plt.tight_layout()
plt.savefig(os.path.join(sig_dir, 'Sample_Signatures_heatmap.pdf'), dpi=300, bbox_inches='tight')
plt.show()
plt.close()

# ---- B) Per-signature boxplots with MWU + FDR ----
n_cols = 4
n_rows = int(np.ceil(len(sig_score_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
axes = axes.flatten()

sig_mw_results = []
for i, sc_name in enumerate(sig_score_cols):
    ax = axes[i]
    sns.boxplot(data=samp_sig, x='group', y=sc_name, order=existing_groups,
                palette=color, width=0.5, ax=ax)
    sns.stripplot(data=samp_sig, x='group', y=sc_name, order=existing_groups,
                   color='black', size=7, jitter=0.12, alpha=0.85, ax=ax)
    
    a_vals = samp_sig.loc[samp_sig['group'] == 'R301', sc_name].values
    b_vals = samp_sig.loc[samp_sig['group'] == 'PBS', sc_name].values
    if len(a_vals) >= 2 and len(b_vals) >= 2:
        u, p = mannwhitneyu(a_vals, b_vals, alternative='two-sided')
        sig_mw_results.append({'signature': sc_name, 'u_stat': u, 'p_value': p})
    else:
        sig_mw_results.append({'signature': sc_name, 'u_stat': np.nan, 'p_value': np.nan})
    
    ax.set_title(sc_name.replace('_', ' '), fontsize=10)
    ax.set_xlabel('')
    ax.set_ylabel('Mean Score per Sample')

for j in range(len(sig_score_cols), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(sig_dir, 'Sample_Signatures_boxplot.pdf'), dpi=300, bbox_inches='tight')
plt.show()
plt.close()

# FDR correction
sig_mw_df = pd.DataFrame(sig_mw_results)
sig_mw_df['p_fdr'] = multipletests(sig_mw_df['p_value'].fillna(1.0).values, method='fdr_bh')[1]
sig_mw_df.to_csv(os.path.join(sig_dir, 'Sample_Signatures_MWU.csv'), index=False)
print('\nSample-Level Signature Comparison (R301 vs PBS):')
print(sig_mw_df.round(4).to_string(index=False))

print(f'\nSample signature figures saved to: {sig_dir}')

### 7.4 Functional Module Scoring Across All Compartments

在 T/NK、髓系、基质等各 compartments 中分别定义功能模块，
计算模块评分并在对应细胞群内比较 PBS vs R301，汇总为全局热图。

In [ ]:
# ==============================================================================
# 7.4 Functional Module Scoring Across All Compartments
# ==============================================================================

import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu

module_dir = os.path.join(RESULTS_DIR, 'compartment_modules')
os.makedirs(module_dir, exist_ok=True)

# --------------------------------------------------------------------------
# 1) Classify cell types into compartments
# --------------------------------------------------------------------------
compartment_map = {
    # T / NK
    'Cytotoxic NK/NKT-like cells': 'T_NK',
    'Cytotoxic CD8 T cells': 'T_NK',
    'Activated T cells': 'T_NK',
    # Myeloid
    'Neutrophils': 'Myeloid',
    'IL1B+ inflammatory macrophages': 'Myeloid',
    'CCL2+ inflammatory macrophages': 'Myeloid',
    'C1q+ antigen-presenting macrophages': 'Myeloid',
    'MRC1+ resident-like macrophages': 'Myeloid',
    'CTSK+ osteoclast-like macrophages': 'Myeloid',
    # DC
    'pDCs': 'Myeloid',
    'CLEC9A+ cDCs': 'Myeloid',
    'CCR7+ mature DCs': 'Myeloid',
    # Stromal
    'ECM-remodeling fibroblasts': 'Stromal',
    'Vascular endothelial cells': 'Stromal',
    # Mast / B
    'Mast cells': 'Myeloid',
    'B cells': 'T_NK',  # small population, group with lymphoid
}

# Malignant cells keep their own fine-grained types
malignant_celltypes = [ct for ct in adata.obs['celltype'].unique() if 'malignant' in ct.lower()
                       or ct.startswith('Prolif') or ct.startswith('Cycling')
                       or ct.startswith('IFN') or ct.startswith('Hypoxic')
                       or ct.startswith('Growth') or ct.startswith('Translation')
                       or ct.startswith('CDH13') or ct.startswith('Mesenchymal')]

adata.obs['compartment'] = adata.obs['celltype'].map(compartment_map)
# Unmapped cell types keep their original name (e.g., malignant subtypes)
adata.obs['compartment'] = adata.obs['compartment'].fillna('Malignant')

print('Compartment distribution:')
print(adata.obs['compartment'].value_counts())

In [ ]:
# --------------------------------------------------------------------------
# 2) Define compartment-specific functional modules
# --------------------------------------------------------------------------

compartment_modules = {
    'T_NK': {
        'Cytotoxicity': ['Gzma', 'Gzmb', 'Gzmk', 'Prf1', 'Ctsw', 'Nkg7', 'Fasl', 'Tnf'],
        'Exhaustion': ['Pdcd1', 'Lag3', 'Havcr2', 'Tigit', 'Ctla4', 'Tox', 'Entpd1'],
        'Activation': ['Cd69', 'Icos', 'Cd44', 'Il2ra', 'Ifng', 'Cd28', 'Cd40lg'],
        'IFN_Response': ['Ifit1', 'Ifit3', 'Isg15', 'Stat1', 'Irf7', 'Irf9', 'Oasl2', 'Rsad2'],
        'Tissue_Residency': ['Itgae', 'Cd69', 'Cxcr6', 'Hobit', 'Runx3', 'Prdm1'],
    },
    'Myeloid': {
        'M1_Polarization': ['Il1b', 'Tnf', 'Nos2', 'Cxcl9', 'Cxcl10', 'Ccl3', 'Cd80', 'Cd86', 'Ciita'],
        'M2_Polarization': ['Mrc1', 'Arg1', 'Chil3', 'Cd163', 'Il10', 'Tgfb1', 'Ccl22', 'Stab1', 'F13a1'],
        'Phagocytosis': ['Fcgr1', 'Fcgr3', 'Mertk', 'Cd68', 'Trem2', 'Itgam', 'Marco', 'Msr1', 'Cd36'],
        'Antigen_Presentation': ['H2-Aa', 'H2-Ab1', 'H2-Eb1', 'Cd74', 'B2m', 'H2-K1', 'H2-D1', 'Tap1', 'Ciita'],
        'Chemokine_Production': ['Ccl2', 'Ccl7', 'Ccl8', 'Ccl12', 'Cxcl1', 'Cxcl2', 'Cxcl3', 'Cxcl9', 'Cxcl10'],
        'Inflammasome': ['Nlrp3', 'Il1b', 'Il18', 'Casp1', 'Pycard', 'Aim2'],
    },
    'Stromal': {
        'ECM_Production': ['Col1a1', 'Col1a2', 'Col3a1', 'Col5a2', 'Fbn1', 'Fn1', 'Postn', 'Tnc'],
        'ECM_Degradation': ['Mmp2', 'Mmp9', 'Mmp14', 'Timp1', 'Timp2', 'Lox', 'Loxl2'],
        'Angiogenesis': ['Pecam1', 'Cdh5', 'Eng', 'Vwf', 'Flt1', 'Kdr', 'Angpt2', 'Nrp1', 'Egfl7'],
        'Inflammatory_CAF': ['Il6', 'Ccl2', 'Cxcl1', 'Cxcl12', 'Ptgs2', 'Lif', 'Csf1'],
    },
}

# For 'Malignant', reuse existing module scores or compute additional ones
# This complements the earlier malignant module scoring (Section 6) with
# treatment-focused modules scored only in malignant cells
compartment_modules['Malignant'] = {
    'EMT_Score': ['Vim', 'Fn1', 'Col1a1', 'Col1a2', 'Prrx1', 'Zeb1', 'Zeb2', 'Snai1', 'Snai2', 'Twist1', 'Cdh13', 'Slit2'],
    'Immune_Evasion': ['Cd274', 'Pdcd1lg2', 'Cd47', 'Serpinb9', 'Fasl', 'Ccl2', 'Cxcl1', 'Vegfa'],
    'Stress_Response': ['Ddit3', 'Hspa5', 'Atf4', 'Ern1', 'Xbp1', 'Hsp90b1', 'Hspa1a', 'Hspa1b', 'Dnajb1'],
}

In [ ]:
# --------------------------------------------------------------------------
# 3) Score modules within each compartment
# --------------------------------------------------------------------------

all_module_results = {}  # {compartment: {module_name: {'PBS_mean': ..., 'R301_mean': ..., 'p_value': ...}}}

for comp, modules in compartment_modules.items():
    # Subset to cells in this compartment
    comp_adata = adata[adata.obs['compartment'] == comp].copy()
    if comp_adata.n_obs < 50:
        print(f'Skipping {comp}: only {comp_adata.n_obs} cells')
        continue
    
    comp_results = {}
    scored_this_comp = []

    for mod_name, genes in modules.items():
        genes_avail = [g for g in genes if g in adata.raw.var_names]
        if len(genes_avail) < 3:
            continue

        mod_score_col = f'{comp}_{mod_name}'
        sc.tl.score_genes(
            comp_adata, gene_list=genes_avail, score_name=mod_score_col,
            use_raw=True, ctrl_size=min(50, len(genes_avail) * 2), random_state=42
        )
        
        # Aggregate to sample level for MWU
        samp_mean = comp_adata.obs.groupby('sampleID')[mod_score_col].mean()
        samp_mean = samp_mean.to_frame().join(sample_to_group)
        
        pbs_vals = samp_mean.loc[samp_mean['group'] == 'PBS', mod_score_col].values
        r301_vals = samp_mean.loc[samp_mean['group'] == 'R301', mod_score_col].values
        
        if len(pbs_vals) >= 2 and len(r301_vals) >= 2:
            u, p = mannwhitneyu(r301_vals, pbs_vals, alternative='two-sided')
        else:
            u, p = np.nan, np.nan
        
        comp_results[mod_name] = {
            'PBS_mean': float(pbs_vals.mean()) if len(pbs_vals) > 0 else np.nan,
            'R301_mean': float(r301_vals.mean()) if len(r301_vals) > 0 else np.nan,
            'delta': float(r301_vals.mean() - pbs_vals.mean()) if (len(r301_vals) > 0 and len(pbs_vals) > 0) else np.nan,
            'p_value': p,
            'n_genes_used': len(genes_avail),
        }
        scored_this_comp.append(mod_name)
    
    all_module_results[comp] = comp_results
    print(f'{comp}: scored {len(scored_this_comp)} modules' + 
          f' ({comp_adata.n_obs} cells)')

print(f'\nTotal compartments scored: {len(all_module_results)}')

In [ ]:
# --------------------------------------------------------------------------
# 4) Comprehensive heatmap of all compartment modules
# --------------------------------------------------------------------------

# Flatten into a dataframe
heatmap_rows = []
for comp, modules in all_module_results.items():
    for mod_name, stats in modules.items():
        heatmap_rows.append({
            'Module': f'{comp}: {mod_name}',
            'Compartment': comp,
            'delta': stats['delta'],
            'p_value': stats['p_value'],
            'PBS_mean': stats['PBS_mean'],
            'R301_mean': stats['R301_mean'],
        })

heat_df = pd.DataFrame(heatmap_rows).set_index('Module')
if not heat_df.empty:
    # FDR correction across all module tests
    from statsmodels.stats.multitest import multipletests
    pvals = heat_df['p_value'].fillna(1.0).values
    _, qvals, _, _ = multipletests(pvals, method='fdr_bh')
    heat_df['q_fdr'] = qvals
else:
    heat_df['q_fdr'] = np.nan

heat_df.to_csv(os.path.join(module_dir, 'Compartment_Modules_Summary.csv'))

# Plot heatmap of delta (R301 - PBS)
plot_data = heat_df[['delta']].copy()
plot_data['Module'] = plot_data.index

fig, ax = plt.subplots(figsize=(6, max(5, 0.4 * len(plot_data) + 1)))

colors = ['#d62728' if d > 0 else '#1f77b4' for d in plot_data['delta']]
bars = ax.barh(range(len(plot_data)), plot_data['delta'], color=colors, edgecolor='white', linewidth=0.5)
ax.set_yticks(range(len(plot_data)))
ax.set_yticklabels(plot_data['Module'], fontsize=8)
ax.invert_yaxis()
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Delta Score (R301 - PBS)', fontsize=12)
ax.set_title('Compartment-Level Functional Module Changes After Treatment',
             fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# Add significance stars
for i, (_, row) in enumerate(heat_df.iterrows()):
    if row['q_fdr'] < 0.001:
        star = '***'
    elif row['q_fdr'] < 0.01:
        star = '**'
    elif row['q_fdr'] < 0.05:
        star = '*'
    else:
        continue
    x_pos = row['delta'] + (0.02 if row['delta'] >= 0 else -0.02)
    ha = 'left' if row['delta'] >= 0 else 'right'
    ax.text(x_pos, i, star, va='center', ha=ha, fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(module_dir, 'Compartment_Modules_delta.pdf'), dpi=300, bbox_inches='tight')
plt.show()
plt.close()

# Print significant modules
print('\nSignificant functional module changes (q < 0.05):')
sig_modules = heat_df[heat_df['q_fdr'] < 0.05]
if len(sig_modules) > 0:
    for idx, row in sig_modules.iterrows():
        direction = 'UP' if row['delta'] > 0 else 'DOWN'
        print(f'  {idx}: delta={row["delta"]:.3f}, q={row["q_fdr"]:.3f} [{direction}]')
else:
    print('  No modules reach significance at q < 0.05 (may be due to n=3 per group)')

print(f'\nCompartment module results saved to: {module_dir}')

In [ ]:
# --------------------------------------------------------------------------
# 5) Per-compartment summary heatmaps (more detailed view)
# --------------------------------------------------------------------------

for comp, modules in all_module_results.items():
    if len(modules) < 2:
        continue
    
    fig, ax = plt.subplots(figsize=(8, max(3, 0.5 * len(modules) + 1)))
    
    mod_names = list(modules.keys())
    deltas = [modules[m]['delta'] for m in mod_names]
    pvals = [modules[m]['p_value'] for m in mod_names]
    pbs_vals = [modules[m]['PBS_mean'] for m in mod_names]
    r301_vals = [modules[m]['R301_mean'] for m in mod_names]
    
    # Delta heatmap-like barhor chart
    vmax = max(abs(min(deltas or [0])), abs(max(deltas or [0])), 0.01) * 1.2
    colors_list = plt.cm.RdBu_r((np.array(deltas) / vmax + 1) / 2)
    
    ax.barh(range(len(mod_names)), deltas, color=colors_list, edgecolor='white', linewidth=0.5)
    ax.set_yticks(range(len(mod_names)))
    ax.set_yticklabels(mod_names, fontsize=10)
    ax.invert_yaxis()
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Delta Score (R301 - PBS)')
    ax.set_title(f'{comp} Compartment: Functional Module Changes',
                 fontsize=12, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    
    # Annotate with p-values
    for i, (m, d, p) in enumerate(zip(mod_names, deltas, pvals)):
        x_pos = d + (vmax * 0.03 if d >= 0 else -vmax * 0.03)
        ha = 'left' if d >= 0 else 'right'
        p_str = f'p={p:.3f}' if not np.isnan(p) else 'n.s.'
        ax.text(x_pos, i, p_str, va='center', ha=ha, fontsize=8, color='#333333')
    
    plt.tight_layout()
    plt.savefig(os.path.join(module_dir, f'Module_delta_{comp}.pdf'), dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

print(f'Per-compartment module figures saved to: {module_dir}')

In [ ]:
# ==============================================================================
# Save updated AnnData with all scores
# ==============================================================================
adata.write(DATA_DIR + '/Step3-sce_annotated.h5ad', compression='gzip')
print('Updated adata with all module/signature scores saved.')

---

### 7.5 Enrichr Cross-Celltype Summary — Global Transcriptional Reprogramming Patterns

汇总所有 cell type 的 Enrichr GO 富集结果，识别 R301 治疗在多种细胞类型中
共同激活或抑制的生物学通路。

In [ ]:
# ==============================================================================
# 7.5 Enrichr Cross-Celltype Summary
# ==============================================================================

import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

enrichr_summary_dir = os.path.join(RESULTS_DIR, 'enrichr_summary')
os.makedirs(enrichr_summary_dir, exist_ok=True)

DEG_DIR = os.path.join(BASE_DIR, 'DEG_Enrichr_all_comparisons/R301_vs_PBS')

def parse_ct(fname):
    return os.path.basename(fname).replace('_Enrichr_up.csv','').replace('_Enrichr_down.csv','').replace('_',' ').replace('  ',' ')

# ---- Load all Enrichr results ----
all_data = []
for direction in ['up', 'down']:
    files = sorted(glob.glob(os.path.join(DEG_DIR, f'*_Enrichr_{direction}.csv')))
    for f in files:
        ct = parse_ct(f)
        df = pd.read_csv(f)
        df['celltype'] = ct
        df['direction'] = direction
        all_data.append(df)

enr_all = pd.concat(all_data, ignore_index=True)
enr_all['Adjusted P-value'] = pd.to_numeric(enr_all['Adjusted P-value'], errors='coerce')
enr_all['-log10q'] = -np.log10(enr_all['Adjusted P-value'] + 1e-300)
enr_sig = enr_all[enr_all['Adjusted P-value'] < 0.05].copy()
print(f'Cell types with Enrichr data: {enr_all["celltype"].nunique()}')
print(f'Total significant enriched terms (q<0.05): {len(enr_sig)}')
print(f'  UP: {(enr_sig["direction"]=="up").sum()}, DOWN: {(enr_sig["direction"]=="down").sum()}')

# ---- Term frequency across cell types ----
term_freq_up = enr_sig[enr_sig['direction']=='up'].groupby('Term')['celltype'].nunique().sort_values(ascending=False)
term_freq_down = enr_sig[enr_sig['direction']=='down'].groupby('Term')['celltype'].nunique().sort_values(ascending=False)

print('\nTop UP terms by cell-type recurrence:')
for t in term_freq_up.head(10).index:
    cts = enr_sig[(enr_sig['Term']==t) & (enr_sig['direction']=='up')]['celltype'].unique()
    print(f'  [{len(cts)} CTs] {t[:80]}')

print('\nTop DOWN terms by cell-type recurrence:')
for t in term_freq_down.head(10).index:
    cts = enr_sig[(enr_sig['Term']==t) & (enr_sig['direction']=='down')]['celltype'].unique()
    print(f'  [{len(cts)} CTs] {t[:80]}')

In [ ]:
# ---- Visualization A: Most recurring terms (UP vs DOWN side-by-side) ----
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for ax, (freq, direction, title, cmap_name) in zip(
    axes,
    [(term_freq_up, 'up', 'UP in R301 vs PBS', 'Reds'),
     (term_freq_down, 'down', 'DOWN in R301 vs PBS', 'Blues')]
):
    top = freq.head(20)
    colors = plt.get_cmap(cmap_name)(np.linspace(0.5, 0.95, len(top)))
    
    ax.barh(range(len(top)), top.values, color=colors, edgecolor='white', linewidth=0.5)
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels([t[:80] for t in top.index], fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel('Number of Cell Types with Significant Enrichment', fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    for i, v in enumerate(top.values):
        ax.text(v + 0.1, i, str(v), va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(enrichr_summary_dir, 'Enrichr_Recurring_Terms_UP_DOWN.pdf'), dpi=300, bbox_inches='tight')
plt.show()
plt.close()
print('Saved: Enrichr_Recurring_Terms_UP_DOWN.pdf')

In [ ]:
# ---- Visualization B: Heatmap of shared terms × cell types ----
for direction, freq_data, cmap_name, direction_label in [
    ('up', term_freq_up, 'Reds', 'UP-regulated'),
    ('down', term_freq_down, 'Blues', 'DOWN-regulated'),
]:
    top_terms = freq_data.head(15).index.tolist()
    heat_data = enr_sig[(enr_sig['Term'].isin(top_terms)) & (enr_sig['direction']==direction)]
    mat = heat_data.pivot_table(
        index='Term', columns='celltype', values='-log10q', aggfunc='max', fill_value=0
    )
    mat = mat.loc[mat.sum(axis=1) > 0, mat.sum(axis=0) > 0]
    if mat.empty:
        continue
    mat.index = [t[:80] for t in mat.index]
    
    fig, ax = plt.subplots(figsize=(max(12, 0.35 * mat.shape[1] + 3), max(6, 0.4 * mat.shape[0] + 2)))
    sns.heatmap(mat, cmap=cmap_name, annot=True, fmt='.1f', linewidths=0.3, linecolor='white',
                cbar_kws={'label': '-log10(Adjusted P-value)'}, ax=ax, annot_kws={'fontsize': 8})
    ax.set_title(f'Shared GO Terms: {direction_label} in R301 vs PBS\n(Top terms by cell-type recurrence)',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel(''); ax.set_ylabel('')
    plt.tight_layout()
    plt.savefig(os.path.join(enrichr_summary_dir, f'Enrichr_SharedTerms_Heatmap_{direction.upper()}.pdf'),
                dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f'Saved: Enrichr_SharedTerms_Heatmap_{direction.upper()}.pdf')

In [ ]:
# ---- Visualization C: Top 3 enriched terms per cell type (dotplot) ----
for direction, title_suffix in [('up', 'UP-regulated'), ('down', 'DOWN-regulated')]:
    top_per_ct = enr_sig[enr_sig['direction'] == direction].copy()
    top_per_ct = top_per_ct.sort_values('Adjusted P-value').groupby('celltype').head(3)
    if top_per_ct.empty:
        continue
    
    top_per_ct['Term_short'] = top_per_ct['Term'].str[:70]
    # Extract hit count from Overlap (e.g., "12/200")
    try:
        top_per_ct['n_genes'] = top_per_ct['Overlap'].astype(str).str.split('/').str[0].astype(float)
    except:
        top_per_ct['n_genes'] = 1.0
    
    unique_terms = top_per_ct.drop_duplicates('Term').sort_values('Adjusted P-value')['Term_short'].tolist()
    celltypes_sorted = sorted(top_per_ct['celltype'].unique())
    ct_to_x = {ct: i for i, ct in enumerate(celltypes_sorted)}
    term_to_y = {t: i for i, t in enumerate(unique_terms)}
    
    fig, ax = plt.subplots(figsize=(12, max(5, 0.35 * len(unique_terms) + 2)))
    scatter = ax.scatter(
        [ct_to_x[ct] for ct in top_per_ct['celltype']],
        [term_to_y[t] for t in top_per_ct['Term_short']],
        s=30 + 12 * top_per_ct['n_genes'].values,
        c=top_per_ct['-log10q'].values,
        cmap='Reds' if direction == 'up' else 'Blues',
        alpha=0.85, edgecolor='grey', linewidth=0.3
    )
    ax.set_xticks(range(len(celltypes_sorted)))
    ax.set_xticklabels(celltypes_sorted, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(unique_terms)))
    ax.set_yticklabels(unique_terms, fontsize=7)
    ax.set_xlim(-0.5, len(celltypes_sorted) - 0.5)
    ax.set_ylim(-0.5, len(unique_terms) - 0.5)
    ax.invert_yaxis()
    cbar = plt.colorbar(scatter, ax=ax, fraction=0.02, pad=0.02)
    cbar.set_label('-log10(Adjusted P-value)')
    ax.set_title(f'Top 3 {title_suffix} GO Terms per Cell Type (R301 vs PBS)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(enrichr_summary_dir, f'Enrichr_Top3_Dotplot_{direction.upper()}.pdf'),
                dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

print(f'\nEnrichr summary saved to: {enrichr_summary_dir}')

In [ ]:
# ---- Save comprehensive cross-celltype summary table ----
summary_rows = []
for direction, freq_data in [('up', term_freq_up), ('down', term_freq_down)]:
    for term, n_cts in freq_data.items():
        cts = enr_sig[(enr_sig['Term']==term) & (enr_sig['direction']==direction)]['celltype'].unique()
        mean_q = enr_sig[(enr_sig['Term']==term) & (enr_sig['direction']==direction)]['Adjusted P-value'].mean()
        summary_rows.append({
            'direction': direction,
            'term': term,
            'n_celltypes': n_cts,
            'celltypes': ', '.join(cts),
            'mean_q': mean_q,
        })

summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.sort_values(['direction', 'n_celltypes', 'mean_q'],
                                      ascending=[True, False, True])
summary_df.to_csv(os.path.join(enrichr_summary_dir, 'Enrichr_CrossCelltype_Summary.csv'), index=False)

print('\n=== Enrichr Cross-Celltype Summary ===')
print(f'UP: Top terms = {term_freq_up.head(5).index.tolist()}')
print(f'DOWN: Top terms = {term_freq_down.head(5).index.tolist()}')
print(f'\nKey conclusion:')
print(f'  R301 broadly UP-regulates IFN/antiviral programs across {term_freq_up.iloc[0]} cell types')
print(f'  R301 broadly DOWN-regulates inflammatory signaling across {term_freq_down.iloc[0]} cell types')

---

## Target Analysis

Analysis of VEGF and PD1 pathway gene expression across cell populations and treatment groups.

### Setup and Population Definition

---

> ⚠️ **以下 Target Analysis / Module Analysis cells (cells 133-135) 包含其他项目的硬编码 group 名**
> (NC, AK112, CD47, RC148)，在当前 PBS vs R301 数据上无法直接运行。
> 如需在当前项目中使用，需替换为 PBS/R301 的 group 名并调整分析逻辑。
> 可参考 Section 7.3 (Sample-Level Gene Signature Scoring) 和 Section 7.4 (Compartment Modules)
> 中已有的治疗响应分析方法。

---


In [ ]:
adata = sc.read(DATA_DIR+ '/Step3-sce_annotated.h5ad')
adata

In [ ]:
# ==============================================================================
# Upgraded VEGF/PD1 Target Analysis Module
# ==============================================================================

import os
import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

# 导入 P 值自动标注工具
try:
    from statannotations.Annotator import Annotator
except ImportError:
    print("⚠️ 未找到 statannotations。请在终端运行: pip install statannotations")
    Annotator = None
# ==============================================================================
# 1. 目录创建与靶点基因定义
# ==============================================================================
target_dir = os.path.join(RESULTS_DIR, 'Target_analysis')
os.makedirs(target_dir, exist_ok=True)

# 定义双抗药物机制相关的核心基因集
target_genes_dict = {
      'VEGF_Pathway': ['Vegfa', 'Vegfb', 'Kdr', 'Flt1', 'Pgf'],
      'PD1_Pathway': ['Cd274', 'Pdcd1', 'Pdcd1lg2'],
      'CD47_Pathway': ['Cd47', 'Sirpa', 'Sirc1'],  # 新增
      'Immune_Checkpoints': ['Ctla4', 'Lag3', 'Havcr2', 'Tigit', 'Cd244'],
      'Angiogenesis': ['Angpt1', 'Angpt2', 'Tek', 'Pdgfa', 'Pdgfb'],
  }

# 展平所有基因并检查在 adata 中的存在情况
all_target_genes = list(set([g for genes in target_genes_dict.values() for g in genes]))
available_genes = [g for g in all_target_genes if g in adata.raw.var_names]
missing_genes = [g for g in all_target_genes if g not in adata.raw.var_names]

print('='*70)
print('🎯 Upgraded VEGF/PD1 Target Analysis')
print('='*70)
print(f'Output directory: {target_dir}')
print(f'Target genes defined: {len(all_target_genes)}')
print(f'Available in data: {len(available_genes)}')
if missing_genes:
    print(f'Missing genes: {missing_genes}')

# ==============================================================================
# 2. 基于样本 (Sample-level) 的精准靶点表达统计
# ==============================================================================
print("\n[Target Analysis] Calculating Sample-level statistics...")

# 精准定义：只在具有生物学意义的细胞群中看特定靶点
specific_targets = {
      'Vegfa': ['Malignant cells', 'Fibroblasts', 'Mrc1+ TAMs', 'Endothelial cells'],
      'Cd274': ['Malignant cells', 'Mrc1+ TAMs', 'cDCs', 'Monocytes'],
      'Pdcd1': ['T cells', 'NK cells'],
      'Havcr2': ['T cells', 'NK cells', 'cDCs'],
      'Kdr': ['Endothelial cells'],
      'Cd47': ['Malignant cells', 'Monocytes', 'Mrc1+ TAMs', 'Ccl8+ TAMs'],  # 配体
      'Sirpa': ['Mrc1+ TAMs', 'Ccl8+ TAMs', 'Monocytes'],  # 受体（巨噬细胞）
  }

sample_stats = []

for gene, target_celltypes in specific_targets.items():
    if gene not in adata.raw.var_names:
        continue
        
    expr = adata.raw[:, gene].X.toarray().flatten()
    
    for celltype in target_celltypes:
        if celltype not in adata.obs['celltype'].values:
            continue
            
        # 按样本(sampleID)级别汇总，用于提供统计学检验所需的独立重复
        for sample in adata.obs['sampleID'].unique():
            # 获取该样本的组别
            sample_df = adata.obs[adata.obs['sampleID'] == sample]
            if sample_df.empty:
                continue
            group = sample_df['group'].iloc[0]
            
            mask = (adata.obs['celltype'] == celltype) & (adata.obs['sampleID'] == sample)
            cell_count = mask.sum()
            
            # 过滤掉细胞数量太少的样本点（避免极小样本造成的百分比伪影）
            if cell_count >= 5: 
                group_expr = expr[mask]
                pct_pos = (group_expr > 0).mean() * 100
                mean_exp = group_expr.mean()
                
                sample_stats.append({
                    'Gene': gene,
                    'CellType': celltype,
                    'Sample': sample,
                    'Group': group,
                    'Cell_Count': cell_count,
                    'Pct_Positive': pct_pos,
                    'Mean_Expr': mean_exp
                })

df_sample_stats = pd.DataFrame(sample_stats)
df_sample_stats.to_csv(os.path.join(target_dir, 'Target_SampleLevel_Stats.csv'), index=False)

# ==============================================================================
# 3. 绘制带有误差棒与 P 值的靶点分布图 (Boxplot + Stripplot)
# ==============================================================================
print("\n[Target Analysis] Generating statistical plots with p-values...")

stat_pairs = [
      ("NC", "AK112"),         # 主效应
      ("NC", "CD47"),          # CD47单药效应
      ("NC", "AK112+CD47"),    # 联合治疗效应
      ("AK112", "AK112+CD47"), # 联合 vs AK112单药
      ("CD47", "AK112+CD47"),  # 联合 vs CD47单药
  ]

for gene in specific_targets.keys():
    if gene not in df_sample_stats['Gene'].values:
        continue
        
    df_plot = df_sample_stats[df_sample_stats['Gene'] == gene]
    if df_plot.empty:
        continue
        
    g = sns.catplot(
        data=df_plot, x='Group', y='Pct_Positive', col='CellType', 
        kind='box', order=treatment_order, palette=group_colors,
        height=4, aspect=0.8, sharey=False
    )
    
    # 添加底层散点，展示真实样本点
    g.map_dataframe(sns.stripplot, x='Group', y='Pct_Positive', 
                    order=treatment_order, color='black', alpha=0.6, jitter=True)
    
    g.set_axis_labels("", f"% {gene}+ Cells")
    g.set_titles("{col_name}")
    g.fig.suptitle(f'{gene} Expression by Sample', y=1.05, fontweight='bold')
    
    # 自动添加显著性检验星号
    if Annotator is not None:
        for ax, celltype in zip(g.axes.flat, df_plot['CellType'].unique()):
            data_subset = df_plot[df_plot['CellType'] == celltype]
            # 确保每组都有数据，防止报错
            if len(data_subset['Group'].unique()) > 1:
                try:
                    annotator = Annotator(ax, stat_pairs, data=data_subset, 
                                          x='Group', y='Pct_Positive', order=treatment_order)
                    annotator.configure(test='t-test_ind', text_format='star', loc='inside')
                    annotator.apply_and_annotate()
                except Exception as e:
                    print(f"  Note: Skipped annotation for {gene} in {celltype} (too few samples)")

    plt.savefig(os.path.join(target_dir, f'Target_{gene}_SampleStats.pdf'), bbox_inches='tight', dpi=300)
    plt.close()

# ==============================================================================
# 4. 嵌套 DotPlot：展示 Celltype x Group 的靶点全景动态
# ==============================================================================
print("\n[Target Analysis] Generating Nested DotPlot...")

# 创建复合列 (例如: "Malignant cells_NC")
adata.obs['celltype_group'] = adata.obs['celltype'].astype(str) + "_" + adata.obs['group'].astype(str)

# 挑选需要重点展示机制响应的核心细胞群
focus_celltypes = ['Malignant cells', 'Resident macrophages', 'Activated macrophages',
    'Activated T cells','Exhausted-like T cells','NK cells', 'Endothelial cells']

ordered_categories = []
for ct in focus_celltypes:
    for grp in treatment_order: 
        ordered_categories.append(f"{ct}_{grp}")

# 过滤出存在于数据中的细胞
valid_categories = [cat for cat in ordered_categories if cat in adata.obs['celltype_group'].unique()]

adata_focus = adata[adata.obs['celltype_group'].isin(valid_categories)].copy()
adata_focus.obs['celltype_group'] = pd.Categorical(adata_focus.obs['celltype_group'], categories=valid_categories, ordered=True)

# 精选一些核心双抗相关基因用于展示
show_genes = [g for g in ['Vegfa', 'Kdr', 'Cd274', 'Pdcd1', 'Havcr2', 'Ctla4', 'Lag3', 'Ifng', 'Gzmb'] if g in available_genes]

if len(show_genes) > 0:
    dp = sc.pl.dotplot(
        adata_focus, 
        var_names=show_genes, 
        groupby='celltype_group', 
        standard_scale='var', 
        cmap='Reds',
        return_fig=True
    )
    dp.style(cmap='Reds', dot_edge_color='black', dot_edge_lw=0.5)
    dp.savefig(os.path.join(target_dir, 'Target_Nested_DotPlot.pdf'), dpi=300, bbox_inches='tight')
    print("  Saved: Target_Nested_DotPlot.pdf")
# ==============================================================================
# 6. 联合治疗协同效应分析 (新增模块)
# ==============================================================================
print("\n[Target Analysis] Analyzing combination therapy synergy...")

def calculate_synergy(ak112_val, cd47_val, combo_val, method='Bliss'):
    """
    计算药物协同效应
    Bliss independence: E_combo = E_A + E_B - E_A * E_B
    如果实际效应 > 预期效应 = 协同 (synergy)
    如果实际效应 < 预期效应 = 拮抗 (antagonism)
    """
    if method == 'Bliss':
        expected = ak112_val + cd47_val - (ak112_val * cd47_val)
        synergy_score = combo_val - expected
        return synergy_score, expected
    return None, None

# 示例：对每个基因在每个细胞类型中计算协同
synergy_results = []

for gene in ['Vegfa', 'Cd274', 'Pdcd1', 'Cd47', 'Ifng', 'Gzmb']:
    if gene not in adata.raw.var_names:
        continue

    for celltype in adata.obs['celltype'].unique():
        # 获取各组的平均表达
        nc_expr = df_sample_stats[(df_sample_stats['Gene']==gene) &
                                (df_sample_stats['CellType']==celltype) &
                                (df_sample_stats['Group']=='NC')]['Mean_Expr'].mean()

        ak112_expr = df_sample_stats[(df_sample_stats['Gene']==gene) &
                                    (df_sample_stats['CellType']==celltype) &
                                    (df_sample_stats['Group']=='AK112')]['Mean_Expr'].mean()

        cd47_expr = df_sample_stats[(df_sample_stats['Gene']==gene) &
                                    (df_sample_stats['CellType']==celltype) &
                                    (df_sample_stats['Group']=='CD47')]['Mean_Expr'].mean()

        combo_expr = df_sample_stats[(df_sample_stats['Gene']==gene) &
                                    (df_sample_stats['CellType']==celltype) &
                                    (df_sample_stats['Group']=='AK112+CD47')]['Mean_Expr'].mean()

        if not any(pd.isna([nc_expr, ak112_expr, cd47_expr, combo_expr])):
            # 计算相对于NC的变化（0-1标准化）
            ak112_effect = (ak112_expr - nc_expr) / nc_expr if nc_expr > 0 else 0
            cd47_effect = (cd47_expr - nc_expr) / nc_expr if nc_expr > 0 else 0
            combo_effect = (combo_expr - nc_expr) / nc_expr if nc_expr > 0 else 0

            synergy_score, expected = calculate_synergy(ak112_effect, cd47_effect, combo_effect)

            synergy_results.append({
                'Gene': gene,
                'CellType': celltype,
                'AK112_Effect': ak112_effect,
                'CD47_Effect': cd47_effect,
                'Combo_Effect': combo_effect,
                'Expected_Effect': expected,
                'Synergy_Score': synergy_score,
                'Synergy_Type': 'Synergy' if synergy_score > 0 else 'Antagonism'
            })

df_synergy = pd.DataFrame(synergy_results)
df_synergy.to_csv(os.path.join(target_dir, 'Synergy_Analysis.csv'), index=False)
# 绘制协同效应热图
if not df_synergy.empty:
    plt.figure(figsize=(10, 6))
    synergy_pivot = df_synergy.pivot(index='Gene', columns='CellType', values='Synergy_Score')
    sns.heatmap(synergy_pivot, cmap='RdBu_r', center=0, annot=True, fmt='.2f',
                cbar_kws={'label': 'Synergy Score (>0 = Synergy)'})
    plt.title('AK112 + CD47 Synergy Analysis', fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(target_dir, 'Synergy_Heatmap.pdf'), dpi=300, bbox_inches='tight')
    plt.close()
# ==============================================================================
# 5. 靶点通路级别 Heatmap (矩阵图)
# ==============================================================================
print("\n[Target Analysis] Generating Pathway Heatmap...")
pathway_genes = [g for g in all_target_genes if g in adata.raw.var_names]

if len(pathway_genes) > 0:
    # 按照 Celltype 分组展示整体特征
    sc.pl.matrixplot(adata, var_names=pathway_genes, groupby='celltype', 
                     cmap='RdBu_r', standard_scale='var',
                     save='Target_pathway_heatmap.pdf')
    print("  Saved: Target_pathway_heatmap.pdf")

print("\n" + "="*70)
print("✅ Target Analysis Module Completed Successfully!")
print("="*70)

### VEGF/PD1 Pathway Module Analysis

Module-based analysis of VEGF signaling, PD1 pathway, angiogenesis, and immune exhaustion.


In [ ]:
# ============================================================
# Upgraded VEGF/PD1 Module-based Analysis
# ============================================================

import os
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# ----------------------------
# USER CONFIG
# ----------------------------
target_dir = os.path.join(RESULTS_DIR, 'Module_Analysis')
os.makedirs(target_dir, exist_ok=True)

min_cells = 10  # 稍微放宽一点，避免某个样本某些细胞过少被剔除
PLOT_ALL_PANEL = True   
q_cut = 0.05

treat_vs_nc_comps = [('AK112', 'NC'), ('RC148', 'NC')]
drug_vs_drug_comps = [('AK112','RC148')]

# ----------------------------
# 1) Merge celltypes (已修正字典映射)
# ----------------------------
myeloid_types = ['Monocytes', 'Mrc1+ TAMs', 'Ccl8+ TAMs', 'Neutrophils', 'cDCs', 'pDCs', 'Mast cells']
t_types = ['T cells'] # 修正为实际存在的标签

def merge_celltype(ct):
    if ct == 'Malignant cells': return 'Tumor'
    if ct in myeloid_types:     return 'Myeloid'
    if ct == 'NK cells':        return 'NK'
    if ct in t_types:           return 'T'
    if ct in ['Fibroblasts', 'Endothelial cells', 'Pericytes']: return 'Stromal'
    return 'Other'

if 'celltype' in adata.obs.columns:
    adata.obs['celltype_merged'] = adata.obs['celltype'].map(merge_celltype).astype('category')
else:
    adata.obs['celltype_merged'] = 'All'

# ----------------------------
# 2) Unified modules (已拆分配体与受体，符合生物学逻辑)
# ----------------------------
modules = {
    "VEGF_Ligand": {
        "score_genes": ['Vegfa','Vegfb','Vegfc','Pgf'],
        "heatmap_genes": ['Vegfa','Vegfb','Pgf'],
        "celltype": "Tumor",  # 肿瘤是配体主要来源
    },
    "VEGF_Receptor": {
        "score_genes": ['Kdr','Flt1','Flt4','Nrp1','Nrp2'],
        "heatmap_genes": ['Kdr','Flt1','Nrp1'],
        "celltype": "Stromal",  # 主要在内皮细胞等基质中
    },
    "PD1_PDL1_Pathway": {
        "score_genes": ['Cd274','Pdcd1','Pdcd1lg2','Ctla4','Cd28','Cd80','Cd86'],
        "heatmap_genes": ['Cd274','Pdcd1','Pdcd1lg2','Ctla4'],
        "celltype": "T",  
    },
    "Angiogenesis": {
        "score_genes": ['Angpt1','Angpt2','Tek','Pdgfa','Pdgfb','Pdgfra','Pdgfrb'],
        "heatmap_genes": ['Angpt1','Angpt2','Tek','Pdgfa','Pdgfb'],
        "celltype": "Stromal",  
    },
    "Immune_Exhaustion": {
        "score_genes": ['Pdcd1','Ctla4','Lag3','Havcr2','Tigit','Cd244','Eomes','Entpd1','Cd39'],
        "heatmap_genes": ['Pdcd1','Ctla4','Lag3','Havcr2','Tigit'],
        "celltype": "T",  
    },
    "Myeloid_Infiltration": {
        "score_genes": ['Itgam','Cd11b','Ly6c','Ly6g','Cxcr2','Ccr2','Cd14','Fcgr1','Fcgr3'],
        "heatmap_genes": ['Itgam','Cd11b','Ly6c','Cxcr2','Ccr2'],
        "celltype": "Myeloid",
    },
}

# ----------------------------
# 3) Compute module scores (cell-level)
# ----------------------------
for m, cfg in modules.items():
    genes = [g for g in cfg["score_genes"] if g in adata.raw.var_names]
    if len(genes) >= 2: # 放宽到2个基因即可打分
        score_col = f"{m}_score"
        sc.tl.score_genes(adata, gene_list=genes, score_name=score_col, use_raw=True)
        print(f"[score] {m}: {len(genes)}/{len(cfg['score_genes'])} genes used")

# ----------------------------
# 4) Upgraded Violin Plot (Cell-level stats + Sample-level dots)
# ----------------------------
def stars(q):
    if q is None or np.isnan(q): return 'ns'
    if q < 0.001: return '***'
    if q < 0.01:  return '**'
    if q < 0.05:  return '*'
    return 'ns'

def violin_with_fdr_stars(ad_sub, score_col, order, palette, comparisons, title, outpath):
    # Cell-level data for violin and stats
    df_cell = ad_sub.obs[['group', 'sampleID', score_col]].dropna().copy()
    
    # Sample-level data for overlay dots
    df_sample = df_cell.groupby(['group', 'sampleID'])[score_col].mean().reset_index()
    
    pvals = []
    for treat, base in comparisons:
        a = df_cell.loc[df_cell['group'] == treat, score_col].values
        b = df_cell.loc[df_cell['group'] == base, score_col].values
        if len(a) > 5 and len(b) > 5:
            p = mannwhitneyu(a, b, alternative='two-sided')[1]
        else:
            p = np.nan
        pvals.append(p)

    qvals = [np.nan]*len(pvals)
    valid = [i for i,p in enumerate(pvals) if not np.isnan(p)]
    if len(valid) > 0:
        q = multipletests([pvals[i] for i in valid], method='fdr_bh')[1]
        for ii, qi in zip(valid, q):
            qvals[ii] = qi

    plt.figure(figsize=(6, 5))
    # 1. 细胞水平的小提琴图 (展示整体转录偏移)
    ax = sns.violinplot(data=df_cell, x='group', y=score_col, order=order, 
                        palette=palette, inner=None, alpha=0.5, linewidth=0)
    
    # 2. 样本水平的大黑点 (展示生物学重复的真实分布)
    sns.stripplot(data=df_sample, x='group', y=score_col, order=order, 
                  color='black', size=8, jitter=0.1, ax=ax)

    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel(f'{score_col} (Cell-level)')
    
    # Add stats lines
    ymin, ymax = ax.get_ylim()
    y0 = ymax + (ymax - ymin) * 0.02
    dy = (ymax - ymin) * 0.08
    x_dict = {g:i for i,g in enumerate(order)}
    
    k = 0
    for (g1, g2), q in zip(comparisons, qvals):
        s = stars(q)
        x1, x2 = x_dict[g1], x_dict[g2]
        y = y0 + k * dy
        ax.plot([x1, x1, x2, x2], [y, y+dy*0.2, y+dy*0.2, y], lw=1.2, c='black')
        ax.text((x1+x2)/2, y+dy*0.25, s, ha='center', va='bottom', fontsize=12)
        k += 1
    ax.set_ylim(ymin, y0 + max(1, k+1) * dy)

    plt.tight_layout()
    plt.savefig(outpath, dpi=300)
    plt.close()

# ----------------------------
# 5) Generate Violins
# ----------------------------
print("\n[Violin] Generating module violins...")
for m, cfg in modules.items():
    score_col = f"{m}_score"
    if score_col not in adata.obs.columns: continue

    for ct in [cfg["celltype"], 'All']:
        ad = adata if ct == 'All' else adata[adata.obs['celltype_merged'] == ct]
        if ad.n_obs < min_cells: continue

        violin_with_fdr_stars(
            ad, score_col, treatment_order, group_colors,
            comparisons=treat_vs_nc_comps,
            title=f"{ct} | {m} (Treat vs NC)",
            outpath=os.path.join(target_dir, f"Violin_{ct}_{m}_TreatVsNC.pdf")
        )

# ----------------------------
# 6) Upgraded Heatmap Helpers (Cell-level stats for delta calculation)
# ----------------------------
def compute_delta_matrix_cell_level(ad_sub, genes, comparison_type):
    # 使用细胞级别的均值差异，并用细胞级数据算p值，保证有统计效力
    deltas, pvals = {}, {}
    base = 'NC' if comparison_type == 'Treatment_vs_NC' else 'AK112'
    comps = ['AK112', 'RC148'] if comparison_type == 'Treatment_vs_NC' else ['RC148']

    expr_df = pd.DataFrame(ad_sub.raw[:, genes].X.toarray(), columns=genes, index=ad_sub.obs_names)
    expr_df['group'] = ad_sub.obs['group'].values

    for treat in comps:
        a = expr_df.loc[expr_df['group'] == treat, genes]
        b = expr_df.loc[expr_df['group'] == base, genes]

        if len(a) > 5 and len(b) > 5:
            deltas[treat] = a.mean(axis=0) - b.mean(axis=0)
            pv = [mannwhitneyu(a[g].values, b[g].values, alternative='two-sided')[1] for g in genes]
            pvals[treat] = pd.Series(pv, index=genes)
        else:
            deltas[treat] = pd.Series([np.nan]*len(genes), index=genes)
            pvals[treat] = pd.Series([np.nan]*len(genes), index=genes)

    delta_df, pval_df = pd.DataFrame(deltas), pd.DataFrame(pvals)
    flat_p = pval_df.values.flatten()
    mask = ~np.isnan(flat_p)
    q = np.full_like(flat_p, np.nan, dtype=float)
    if mask.sum() > 0:
        q[mask] = multipletests(flat_p[mask], method='fdr_bh')[1]
    qval_df = pd.DataFrame(q.reshape(pval_df.shape), index=pval_df.index, columns=pval_df.columns)
    
    return delta_df, pval_df, qval_df

def plot_delta_heatmap(delta_df, title, outpath, qval_df, q_cut):
    # 修复了 applymap 在新版 pandas 中的 deprecation 警告，改用 map
    ann_str = delta_df.apply(lambda col: col.map(lambda x: "" if pd.isna(x) else f"{x:.2f}"))
    sig_mask = (qval_df < q_cut)
    
    for r in ann_str.index:
        for c in ann_str.columns:
            if sig_mask.loc[r, c] and ann_str.loc[r, c] != "":
                ann_str.loc[r, c] = ann_str.loc[r, c] + "*"

    plt.figure(figsize=(max(4, 0.95*delta_df.shape[1] + 3), max(3, 0.50*delta_df.shape[0] + 2)))
    ax = sns.heatmap(delta_df, cmap='vlag', center=0, linewidths=0.5,
                     annot=ann_str, fmt="", cbar_kws={'label': 'Delta (log1p Expr)'})
    ax.set_title(title, pad=15)
    ax.set_ylabel('')
    plt.tight_layout()
    plt.savefig(outpath, dpi=300)
    plt.close()

# ----------------------------
# 7) Generate Heatmaps
# ----------------------------
print("\n[Heatmap] Generating delta heatmaps...")
for m, cfg in modules.items():
    genes = [g for g in cfg["heatmap_genes"] if g in adata.raw.var_names]
    if len(genes) < 2: continue

    ad = adata[adata.obs['celltype_merged'] == cfg["celltype"]].copy()
    if ad.n_obs < min_cells: continue

    for comp_type in ['Treatment_vs_NC', 'RC148_vs_AK112']:
        delta_df, pval_df, qval_df = compute_delta_matrix_cell_level(ad, genes, comp_type)
        out = os.path.join(target_dir, f"Heatmap_{m}_{cfg['celltype']}_{comp_type}.pdf")
        plot_delta_heatmap(delta_df, f"{m} ({cfg['celltype']}) | {comp_type}", out, qval_df, q_cut)

print("\n✅ Module-based analysis successfully finished!")

---

## Export & Split

### Load Cleaned Data

In [ ]:
adata = sc.read(DATA_DIR+ '/Step1-sce_cleaned.h5ad')
#adata = sc.read(DATA_DIR+ '/Step0-combined_raw_data.h5ad')
adata

### Check Naming

In [ ]:
# Check cell naming
print("\nadata cell names:")
print(adata.obs_names[:5])

### Load Metadata

In [ ]:
meta = pd.read_csv(RESULTS_DIR + "/sce_metadata_all.csv", index_col=0, sep="\t")
meta.head()

### Distribution Check

In [ ]:
ctab = pd.crosstab(meta['celltype'],meta['group'],margins=True)
print(ctab)

### Subset Cells

In [ ]:
# target_cells = meta[meta["celltype"].isin(["Endothelial cells"])].index
cell_types = meta["celltype"].unique().tolist()
target_cells = meta[meta["celltype"].isin(cell_types)].index
sub = adata[
    adata.obs.index.isin(target_cells), :
]  # Subset adata based on these indices
# Assign the 'celltype' column from meta to NKT.obs
sub.obs["celltype"] = meta.loc[sub.obs.index, "celltype"]
sub.obs["group"] = meta.loc[sub.obs.index, "group"]
sub.obs["batch"] = meta.loc[sub.obs.index, "batch"]
# sub.obs["Main_celltype"] = meta.loc[sub.obs.index, "Main_celltype"]
# Save the result
# Print information about the subset
print(f"Subset contains {sub.n_obs} cells and {sub.n_vars} genes")
sub
# sub.write_h5ad(DATA_DIR + "/Step4-sce_hippo_microglia.h5ad", compression="gzip")#

### Gene Info

In [ ]:
# Readgene_info_has.txt文件（假设路径正确，使用TAB分隔）
gene_info = pd.read_csv('/Users/luye/Data/Sig/gene_info_mmu.txt', sep='\t')

# Filter protein_coding，并获取唯一的external_gene_name（假设你的var_names是基因符号，如'MT-ND1'）
protein_coding_genes = gene_info[gene_info['gene_biotype'] == 'protein_coding']['external_gene_name'].dropna().unique().tolist()
print(f"Found {len(protein_coding_genes)} protein-coding genes.")

# 如果你的var_names是ensembl_gene_id，改用：
# protein_coding_genes = gene_info[gene_info['gene_biotype'] == 'protein_coding']['ensembl_gene_id'].dropna().unique().tolist()

# Filter adata，只保留这些基因
sub = sub[:, sub.var_names.isin(protein_coding_genes)]
print(f"After filtering: {sub.shape[1]} genes remaining.")

### Export Matrix

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np

# Readh5ad文件
#adata = sc.read_h5ad("文件路径.h5ad")

# 导出表达矩阵（稀疏矩阵转为CSR格式）
expr_matrix = sub.X.T
from scipy.io import mmwrite
mmwrite("raw_counts_matrix.mtx", expr_matrix)

### Export Genes

In [ ]:
# Export gene info
pd.DataFrame(sub.var_names).to_csv("genes.csv", index=False, header=False)

### Export Metadata

In [ ]:
# Export cell metadata
sub.obs.to_csv("cell_metadata.csv")

### Export UMAP

In [ ]:
bdata = sc.read(DATA_DIR+ '/Step3-sce_annotated.h5ad')
# Export UMAP coords
umap_coords = pd.DataFrame(bdata.obsm['X_umap_harmony'], index=bdata.obs_names)
umap_coords.columns = ['UMAP_1', 'UMAP_2']
umap_coords.to_csv("umap_coords.csv")

In [ ]:
sc.logging.print_header()